# ECM Neighborhood Analysis v4

In [ ]:
import scanpy
import squidpy
import sklearn
import statsmodels

print(f"scanpy: {scanpy.__version__}")
print(f"squidpy: {squidpy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"statsmodels: {statsmodels.__version__}")

In [ ]:
# Autoreload extension for IPython/Jupyter
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except (ImportError, NameError):
    pass

import pandas as pd
import numpy as np
import anndata as ad
import s3fs

import scanpy as sc
import matplotlib.pyplot as plt
import openslide
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter, AllDataCenters

sc.set_figure_params(dpi=150, color_map="viridis_r")
sc.settings.verbosity = 0

from cloudpathlib import S3Path
from scipy.sparse import issparse
from joblib import Parallel, delayed
from tqdm import tqdm

from gbmcollaboration.definitions import (
    DATA_PATH_CHUV_FILES,
    DATA_PATH_UKER_FILES,
    DATA_PATH_CR_FILES,
)
from gbmcollaboration import AllDataCenters

### Load clinical data

In [ ]:
import pandas as pd
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter

# Load processed clinical tables for all centers and combine into one cohort

def load_processed_clinical(dc, center_name):
    d = dc.load_tabular(exclude_idh1_mut=False)
    clin = d["clinical"]
    key = next((k for k in clin if "processed" in str(k).lower() and "clinical" in str(k).lower()), None)
    df = clin[key].copy() if key else list(clin.values())[0].copy()
    df["center"] = center_name
    return df

ch = load_processed_clinical(CHUVDataCenter, "CHUV")
uk = load_processed_clinical(UKERDataCenter, "UKER")
cr = load_processed_clinical(CRDataCenter, "Charite")

clin_all = pd.concat([ch, uk, cr], ignore_index=True, sort=False)
clin_out = clin_all.copy()

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

### Load Visium data

In [ ]:
adata = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    list_sample_id=None,
    normalization="log_norm_count",
    radius=1,
    deconvolution_slot_name="deconv_cohort_level_4",
)

print(adata)

### Cell2Location loader

In [ ]:
# Load ALL C2L abundance data (CHUV, UKER, Charite) - COMBINED
import gc
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor

def load_sample(center_name, sample_id, h5ad_file):
    """Load and process a single sample using Cell2LocationLoader approach."""
    try:
        if isinstance(h5ad_file, S3Path) or str(h5ad_file).startswith('s3://'):
            s3 = s3fs.S3FileSystem(anon=False)
            h5ad_path_str = str(h5ad_file).replace('s3://', '')
            with s3.open(h5ad_path_str, 'rb') as f:
                adata = ad.read_h5ad(f)
        else:
            adata = ad.read_h5ad(h5ad_file)
        
        if 'means_cell_abundance_w_sf' not in adata.obsm:
            return None
        
        celltype_names = None
        if 'mod' in adata.uns and 'factor_names' in adata.uns['mod']:
            celltype_names = adata.uns['mod']['factor_names']
        elif 'factor_names' in adata.uns:
            celltype_names = adata.uns['factor_names']
        elif 'cell_type_names' in adata.uns:
            celltype_names = adata.uns['cell_type_names']
        
        if celltype_names is None:
            n_celltypes = adata.obsm['means_cell_abundance_w_sf'].shape[1]
            celltype_names = [f'celltype_{i}' for i in range(n_celltypes)]
        
        celltype_names = [f'meanscell_abundance_w_sf_{name}' for name in celltype_names]
        
        cell_abundance = adata.obsm['means_cell_abundance_w_sf']
        if issparse(cell_abundance):
            cell_abundance = cell_abundance.toarray()
        
        df = pd.DataFrame(cell_abundance, index=adata.obs_names, columns=celltype_names)
        df = df.fillna(0)
        df.index = pd.MultiIndex.from_tuples(
            [(sample_id, idx) for idx in df.index],
            names=['sample', 'barcode']
        )
        df['total_cells'] = df.sum(axis=1)
        
        return df
    except Exception as e:
        return None


def load_center(center_name, cell2loc_path):
    """Load one center with original fast parallel, then kill workers."""
    print(f"\nProcessing {center_name}...")
    base_path = S3Path(str(cell2loc_path))
    sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
    print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")
    
    samples_to_process = [
        (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
        for sample_id in sample_dirs
    ]
    
    # Original fast parallel call
    center_results = Parallel(n_jobs=-1)(
        delayed(load_sample)(center_name, sample_id, h5ad_file)
        for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
    )
    
    center_results = [df for df in center_results if df is not None]
    
    for df in center_results:
        new_index = pd.MultiIndex.from_tuples(
            [(center_name, *idx) for idx in df.index],
            names=["center", "sample", "barcode"],
        )
        df.index = new_index
        df["center"] = center_name
    
    df_center = pd.concat(center_results, axis=0)
    
    print(f"  Loaded {df_center.index.get_level_values('sample').nunique()} samples from {center_name}")
    print(f"  Total spots: {df_center.shape[0]}")
    
    # FORCE KILL parallel workers to prevent hang on next call
    get_reusable_executor().shutdown(wait=True)
    gc.collect()
    
    return df_center


# Load each center (workers killed between each)
df_chuv = load_center("CHUV", DATA_PATH_CHUV_FILES["Cell2location"])
df_uker = load_center("UKER", DATA_PATH_UKER_FILES["Cell2location"])
df_cr = load_center("CR", DATA_PATH_CR_FILES["Cell2location"])


# Merge all centers
df_all = pd.concat([df_chuv, df_uker, df_cr], axis=0)

print("\n" + "=" * 60)
print("Combined df_all summary:")
print("  Total spots:", df_all.shape[0])
print("  Total samples:", df_all.index.get_level_values("sample").nunique())
print("  Index levels:", df_all.index.names)
print("  Example columns:", df_all.columns[:5].tolist())
print("=" * 60)

### Merge C2L data

In [ ]:
# Drop the 'center' level from df_all's index for alignment
if isinstance(df_all.index, pd.MultiIndex) and "center" in df_all.index.names:
    df_all_no_center = df_all.copy()
    df_all_no_center.index = df_all_no_center.index.droplevel("center")
else:
    df_all_no_center = df_all

print("df_all_no_center index names:", df_all_no_center.index.names)
print("df_all_no_center shape:", df_all_no_center.shape)

# Build (sample, barcode) keys for each obs row, in obs order
barcodes = pd.Index(adata.obs_names).str.split("_").str[-1]
keys = pd.MultiIndex.from_arrays(
    [adata.obs["sample"].values, barcodes.values],
    names=["sample", "barcode"],
)

print("First 3 keys:", list(keys[:3]))
print("Number of keys:", len(keys))

# Align df_all_no_center to these keys
df_all_aligned = df_all_no_center.reindex(keys)
print("df_all_aligned shape:", df_all_aligned.shape)

# Drop columns from df_all_aligned that already exist in adata.obs
cols_to_add = [c for c in df_all_aligned.columns if c not in adata.obs.columns]
df_to_add = df_all_aligned[cols_to_add].copy()
df_to_add.index = adata.obs.index

# Join instead of concat to avoid fragmentation
adata.obs = adata.obs.join(df_to_add)

print(f"\nNumber of added columns: {len(cols_to_add)}")
print(f"Skipped (already exist): {len(df_all_aligned.columns) - len(cols_to_add)}")
print(f"adata.obs shape: {adata.obs.shape}")

### Load custom cell annotations from file

In [ ]:
import pandas as pd

# user vars
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

# 1) Load key
key_raw = pd.read_csv(KEY_PATH)

# 2) Basic column checks
required_cols = [LEVEL4_KEY] + NEW_COLS
missing_cols = [c for c in required_cols if c not in key_raw.columns]
if missing_cols:
    raise ValueError(f"Missing columns in key file: {missing_cols}")

# 3) Deduplicate on the level4 key and standardize to string
key_unique = key_raw.drop_duplicates(subset=[LEVEL4_KEY]).copy()
key_unique[LEVEL4_KEY] = key_unique[LEVEL4_KEY].astype(str)

# 4) Build a tidy DataFrame indexed by level4
celltype_key = key_unique.set_index(LEVEL4_KEY)[NEW_COLS]
celltype_key = celltype_key.apply(lambda s: s.astype(str))

# 5) Attach as the canonical celltype hierarchy in adata.uns
adata.uns["celltype_annotation_key"] = celltype_key

# 6) Quick coverage checks vs Visium abundance columns
lvl4_in_key = pd.Index(celltype_key.index)

obs_cols = pd.Index(adata.obs.columns)

# deconv_cohort_level_4 columns should use the raw level4 names
deconv_level4_cols = obs_cols.intersection(lvl4_in_key)

# C2L totals use "meanscell_abundance_w_sf_<level4>"
c2l_cols = [c for c in obs_cols if c.startswith("meanscell_abundance_w_sf_")]
c2l_lvl4 = pd.Index([c.replace("meanscell_abundance_w_sf_", "") for c in c2l_cols])
c2l_in_key = c2l_lvl4.intersection(lvl4_in_key)

missing_for_deconv = lvl4_in_key.difference(deconv_level4_cols)
missing_for_c2l = lvl4_in_key.difference(c2l_in_key)

print("Stored celltype_annotation_key in adata.uns['celltype_annotation_key']")
print(f"Total level4 types in key: {len(lvl4_in_key)}")
print(f"Matching deconv level4 columns in adata.obs: {len(deconv_level4_cols)}")
print(f"Matching C2L level4 types (from meanscell_abundance_w_sf_*): {len(c2l_in_key)}")

if len(missing_for_deconv):
    print(f"Level4 types in key not seen as deconv columns: {len(missing_for_deconv)}")
if len(missing_for_c2l):
    print(f"Level4 types in key not seen in C2L columns: {len(missing_for_c2l)}")

### Define ECM genes in matrisome list

In [ ]:
# %% [markdown]
# ### Define brain/GBM ECM gene panel and assess expression/heterogeneity

# %%
import numpy as np
import pandas as pd
from scipy.sparse import issparse

# Core ECM + remodelers + matricellular panel for brain/GBM
ecm_genes_brain = [
    # Collagens
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
    "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
    "COL7A1", "COL8A1", "COL11A1", "COL12A1",
    "COL14A1", "COL15A1", "COL18A1",
    "COL10A1", # "COL16A1", 

    # ECM glycoproteins
    "FN1", "FBN1", "FBN2",
    "TNC", "TNR", "TNXB",
    "SPARC", "POSTN", "SPP1", "EFEMP1",
    "LAMA1", "LAMA2", "LAMA4", "LAMA5",
    "LAMB1", "LAMB2", "LAMB3", 
    "LAMC1", "LAMC3",
    "THBS1", "THBS2", "THBS4",
    "VTN", 
    "NID1", "NID2",
    "FBLN1", "FBLN2", "FBLN5",
    "MMRN2", "VWA1", "TGFBI", "LTBP1",
    "CHI3L1", # "IGFBP7",
    "LGALS1", # "LGALS3",
    "CCN1", "CCN2",
    "EMILIN1",

    # Proteoglycans
    "VCAN", "ACAN", "BCAN", "NCAN",
    "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN",
    "HSPG2", "AGRN",
    "GPC1", "GPC2", "GPC3",
    "SDC1", "SDC4",
    "CSPG4", "CSPG5", "PTPRZ1",
    "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",

    # Proteases
    "MMP2", "MMP7", "MMP9", "MMP11","MMP12", "MMP13", "MMP14", "MMP16",
    "ADAMTS1", "ADAMTS4", "ADAMTS5", 
    "ADAM8",
    "HPSE",
    "PLAU", "PLAT",
    "CTSB", # "CTSD",

    # Protease inhibitors
    "TIMP1", "TIMP2", "TIMP3",
    "SERPINE1",

    # Crosslinking / collagen maturation enzymes
    "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4",  # restored (4 patients)
    "TGM2",
    "P4HA1", "P4HA2",
    "PLOD1", "PLOD2", # "PLOD3",
    "BMP1", "PCOLCE",

    # Hyaluronan metabolism
    "HAS2", "HAS3",
    "HYAL2",

    # Optional inclusions
    # "IGFBP2", 
    "IGFBP5",  # GBM-associated secreted ECM factors
    "SRGN",    # secreted proteoglycan, myeloid/mast ECM
]

# Use the normalization layer if present, otherwise fallback to X
if hasattr(adata, "layers") and "log_norm_count" in adata.layers:
    X_source = adata.layers["log_norm_count"]
else:
    X_source = adata.X

if issparse(X_source):
    X_source = X_source.tocsr()

var_names = adata.var_names

# Determine which genes are present in the AnnData
ecm_genes_unique = sorted(set(ecm_genes_brain))
present_genes = [g for g in ecm_genes_unique if g in var_names]
missing_genes = [g for g in ecm_genes_unique if g not in var_names]

print(f"Total ECM genes in panel: {len(ecm_genes_unique)}")
print(f"Present in adata: {len(present_genes)}")
print(f"Missing from adata: {len(missing_genes)}")

if missing_genes:
    print("Example missing genes:", missing_genes[:10])

# Extract submatrix for present genes (spots x genes)
X_sub = adata[:, present_genes].X if X_source is adata.X else X_source[:, [var_names.get_loc(g) for g in present_genes]]

if issparse(X_sub):
    X_sub = X_sub.toarray()

n_spots = adata.n_obs
samples = adata.obs["sample"].to_numpy()
unique_samples, sample_inverse = np.unique(samples, return_inverse=True)
n_samples = len(unique_samples)
spots_per_sample = np.bincount(sample_inverse)

# Boolean expression mask (non-zero in the chosen layer)
expr_mask = X_sub > 0

metrics = []

for j, gene in enumerate(present_genes):
    vals = X_sub[:, j]
    expr = expr_mask[:, j]

    n_expr = int(expr.sum())
    pct_expr = (n_expr / n_spots) * 100.0

    mean_all = float(vals.mean())
    std_all = float(vals.std())
    mean_nonzero = float(vals[expr].mean()) if n_expr > 0 else 0.0
    cv_all = std_all / (mean_all + 1e-8)

    # Per-sample expression statistics
    expr_per_sample = np.bincount(sample_inverse, weights=expr.astype(np.int64))
    mean_expr_per_sample = np.bincount(sample_inverse, weights=vals) / (spots_per_sample + 1e-8)
    frac_expr_per_sample = expr_per_sample / (spots_per_sample + 1e-8)

    n_samples_ge_5pct = int((frac_expr_per_sample >= 0.05).sum())
    mean_frac_expr_per_sample = float(frac_expr_per_sample.mean())
    cv_sample_mean_expr = float(mean_expr_per_sample.std() / (mean_expr_per_sample.mean() + 1e-8))

    metrics.append(
        {
            "gene": gene,
            "n_spots_expr": n_expr,
            "pct_spots_expr": pct_expr,
            "mean_expr_all_spots": mean_all,
            "mean_expr_nonzero_spots": mean_nonzero,
            "cv_expr_all_spots": cv_all,
            "n_samples_ge_5pct_spots_expr": n_samples_ge_5pct,
            "n_samples_total": n_samples,
            "mean_frac_spots_expr_per_sample": mean_frac_expr_per_sample,
            "cv_mean_expr_per_sample": cv_sample_mean_expr,
        }
    )

ecm_expr_summary = (
    pd.DataFrame(metrics)
    .set_index("gene")
    .sort_values("pct_spots_expr", ascending=False)
)

print("\nECM expression summary (head):")
print(ecm_expr_summary.head(20))

### Leiden clustering based on matrisome

In [ ]:
import numpy as np
import scanpy as sc
from scipy import sparse
import time

# Parameters
ecm_cluster_key   = "ECM_leiden"
neighbors_key     = "ecm_neighbors"
ecm_rep_key       = "X_ecm"
umap_key          = "X_umap_ECM"

leiden_resolution  = 0.6
n_neighbors        = 20
ecm_umap_min_dist  = 0.5
ecm_umap_spread    = 1.2
LAYER              = "log_norm_count"
metric             = "euclidean"
random_state       = 0

adata_target = adata
t_start = time.time()

# 1) Select ECM genes present in this object
ecm_genes_filtered = [g for g in ecm_genes_brain if g in adata_target.var_names]
print(f"ECM genes: {len(ecm_genes_brain)} panel, {len(ecm_genes_filtered)} present")

# 2) Extract matrix
if LAYER is not None and LAYER in adata_target.layers:
    X_layer = adata_target.layers[LAYER]
else:
    X_layer = adata_target.X

idxs = adata_target.var_names.get_indexer(ecm_genes_filtered)
X_ecm = X_layer[:, idxs]
if sparse.issparse(X_ecm):
    X_ecm = X_ecm.toarray()
else:
    X_ecm = np.asarray(X_ecm)

# 3) Drop zero-variance genes
gene_var = X_ecm.var(axis=0)
keep = gene_var > 0
if keep.sum() < len(ecm_genes_filtered):
    dropped = [g for g, k in zip(ecm_genes_filtered, keep) if not k]
    print(f"Dropping {len(dropped)} zero-variance genes: {dropped}")
    X_ecm = X_ecm[:, keep]
    ecm_genes_filtered = [g for g, k in zip(ecm_genes_filtered, keep) if k]

# 4) Per-sample z-scoring
samples = adata_target.obs["sample"].to_numpy()
unique_samples, sample_inverse = np.unique(samples, return_inverse=True)

X_ecm_z = np.zeros_like(X_ecm, dtype=np.float32)
for i in range(len(unique_samples)):
    mask = (sample_inverse == i)
    X_s = X_ecm[mask]
    mu = X_s.mean(axis=0)
    sd = X_s.std(axis=0)
    sd[sd == 0] = 1.0
    X_ecm_z[mask] = (X_s - mu) / sd

# 5) Post-z-score NaN check
nan_mask = np.isnan(X_ecm_z).any(axis=0)
if nan_mask.any():
    dropped = [g for g, k in zip(ecm_genes_filtered, nan_mask) if k]
    print(f"Dropping {len(dropped)} NaN genes: {dropped}")
    X_ecm_z = X_ecm_z[:, ~nan_mask]
    ecm_genes_filtered = [g for g, k in zip(ecm_genes_filtered, ~nan_mask) if k]

# 6) Store ECM representation
adata_target.obsm[ecm_rep_key] = X_ecm_z
adata_target.uns["ecm_genes_used"] = ecm_genes_filtered
print(f"ECM feature matrix shape: {X_ecm_z.shape}, elapsed: {time.time()-t_start:.0f}s")

# 7) Neighbors
t1 = time.time()
print("Building neighbors...")
sc.pp.neighbors(adata_target, use_rep=ecm_rep_key, n_neighbors=n_neighbors,
                key_added=neighbors_key, metric=metric, random_state=random_state)
print(f"  Neighbors done: {time.time()-t1:.0f}s")

# 8) Leiden (igraph for speed on large datasets)
t2 = time.time()
print("Running Leiden (igraph)...")
sc.tl.leiden(adata_target, resolution=leiden_resolution, key_added=ecm_cluster_key,
             neighbors_key=neighbors_key, flavor="igraph", n_iterations=2)
print(f"  Leiden done: {time.time()-t2:.0f}s")

# 9) UMAP
t3 = time.time()
print("Running UMAP...")
sc.tl.umap(adata_target, neighbors_key=neighbors_key, min_dist=ecm_umap_min_dist,
           spread=ecm_umap_spread, random_state=random_state)
adata_target.obsm[umap_key] = adata_target.obsm["X_umap"]
print(f"  UMAP done: {time.time()-t3:.0f}s")

n_clust = adata_target.obs[ecm_cluster_key].nunique()
print(f"\nTotal time: {time.time()-t_start:.0f}s, Clusters: {n_clust}")
print(adata_target.obs[ecm_cluster_key].value_counts().sort_index())

### Save cluster data to file

In [ ]:
# %% [markdown]
# ### Save ECM clustering + UMAP to disk

# %%
import os
import json
import numpy as np
import pandas as pd

RESULTS_DIR    = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"
RESULTS_PREFIX = "ecm_neighborhoods_v4"  # change if you make new versions

# pick AnnData
if "adata_target" in locals():
    ad = adata_target
elif "adata" in locals():
    ad = adata
else:
    raise ValueError("No AnnData object found (adata_target or adata).")

cluster_key = "ECM_leiden"
umap_key    = "X_umap_ECM"
ecm_rep_key = "X_ecm"
neighbors_key = "ecm_neighbors"

# sanity checks
if cluster_key not in ad.obs.columns:
    raise KeyError(f"obs['{cluster_key}'] not found. Run ECM clustering first.")
if umap_key not in ad.obsm:
    raise KeyError(f"obsm['{umap_key}'] not found. Run ECM UMAP first.")

os.makedirs(RESULTS_DIR, exist_ok=True)

# 1) Save per-spot cluster + UMAP as CSV
df = pd.DataFrame(index=ad.obs_names)
df[cluster_key] = ad.obs[cluster_key].astype(str)

umap = np.asarray(ad.obsm[umap_key])
if umap.shape[1] < 2:
    raise ValueError(f"{umap_key} has shape {umap.shape}, expected at least 2 columns.")

df["ECM_UMAP1"] = umap[:, 0]
df["ECM_UMAP2"] = umap[:, 1]

csv_path = os.path.join(RESULTS_DIR, f"{RESULTS_PREFIX}.csv")
df.to_csv(csv_path)
print("Saved ECM clusters + UMAP to:", csv_path)

# 2) Save metadata (ECM genes used, keys, etc.) as JSON
meta = {
    "ecm_genes_used": list(ad.uns.get("ecm_genes_used", [])),
    "cluster_key": cluster_key,
    "umap_key": umap_key,
    "ecm_rep_key": ecm_rep_key,
    "neighbors_key": neighbors_key,
}

json_path = os.path.join(RESULTS_DIR, f"{RESULTS_PREFIX}_meta.json")
with open(json_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved ECM metadata to:", json_path)

### Reload clustering from file

In [ ]:
# %% [markdown]
# ### Load ECM clustering + UMAP from disk and merge into AnnData

# %%
import os
import json
import numpy as np
import pandas as pd

RESULTS_DIR    = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"
RESULTS_PREFIX = "ecm_neighborhoods_v4"

csv_path  = os.path.join(RESULTS_DIR, f"{RESULTS_PREFIX}.csv")
meta_path = os.path.join(RESULTS_DIR, f"{RESULTS_PREFIX}_meta.json")

if not os.path.exists(csv_path):
    raise FileNotFoundError(csv_path)
if not os.path.exists(meta_path):
    raise FileNotFoundError(meta_path)

# pick AnnData
if "adata_target" in locals():
    ad = adata_target
elif "adata" in locals():
    ad = adata
else:
    raise ValueError("No AnnData object found (adata_target or adata). Load your Visium cohort first.")

# load results
df = pd.read_csv(csv_path, index_col=0)
with open(meta_path, "r") as f:
    meta = json.load(f)

cluster_key = meta.get("cluster_key", "ECM_leiden")
umap_key    = meta.get("umap_key", "X_umap_ECM")

# sanity checks on index alignment
missing_in_df   = ad.obs_names.difference(df.index)
missing_in_adata = df.index.difference(ad.obs_names)

print(f"Spots in AnnData: {ad.n_obs}")
print(f"Rows in results:  {df.shape[0]}")
print(f"Spots missing in results: {len(missing_in_df)}")
print(f"Rows not found in AnnData: {len(missing_in_adata)}")

if len(missing_in_df) or len(missing_in_adata):
    print("Warning: mismatched indices detected. Proceeding with intersection only.")

# align df to AnnData obs order
df_aligned = df.reindex(ad.obs_names)

# 1) Attach ECM clusters
ad.obs[cluster_key] = df_aligned[cluster_key].astype("category")

# 2) Attach ECM UMAP
umap_cols = ["ECM_UMAP1", "ECM_UMAP2"]
if not all(c in df_aligned.columns for c in umap_cols):
    raise KeyError(f"Expected columns {umap_cols} in results CSV.")

umap_array = df_aligned[umap_cols].to_numpy(dtype=float)
ad.obsm[umap_key] = umap_array

# 3) Restore ECM gene list in .uns (optional but nice to have)
if "ecm_genes_used" in meta:
    ad.uns["ecm_genes_used"] = meta["ecm_genes_used"]

print(f"Restored obs['{cluster_key}'] and obsm['{umap_key}'] onto AnnData.")
print(ad.obs[cluster_key].value_counts().sort_index())
print("UMAP shape:", ad.obsm[umap_key].shape)

### Visualize ECM UMAP

In [ ]:
### UMAP of ECM-based clusters

# %%
import matplotlib.pyplot as plt
import scanpy as sc

# plotting parameters
plot_width      = 11
plot_height     = 9
point_size      = 4          # lower is faster for big cohorts
show_labels     = True
color_by        = "ECM_leiden"   # or any obs column / gene name
ecm_cluster_key = "ECM_leiden"
color_map       = "tab20"    # used only if color_by == ECM_leiden
vmin            = None
vmax            = None
excluded_clusters = []       # e.g. ["0", "1"]

# choose base AnnData
if "adata_target" in locals():
    adata_base = adata_target
elif "adata_ecm" in locals():
    adata_base = adata_ecm
elif "adata" in locals():
    adata_base = adata
else:
    raise ValueError("No data object available for plotting.")

# choose which UMAP to use
if "X_umap_ECM" in adata_base.obsm:
    basis = "X_umap_ECM"
elif "X_umap" in adata_base.obsm:
    basis = "X_umap"
else:
    raise ValueError("No UMAP embedding found. Run the ECM UMAP cell first.")

# optional filtering by ECM cluster
if ecm_cluster_key in adata_base.obs.columns and excluded_clusters:
    mask = ~adata_base.obs[ecm_cluster_key].isin(excluded_clusters)
    adata_plot = adata_base[mask]
else:
    adata_plot = adata_base

# palette and legend
palette = color_map if color_by == ecm_cluster_key else None

if color_by == ecm_cluster_key:
    legend_loc = "on data" if show_labels else "right margin"
else:
    # for genes / continuous values a side legend or colorbar is fine
    legend_loc = "right margin"

title = f"ECM UMAP: {color_by}"
if excluded_clusters:
    title += " (excluded: " + ", ".join(excluded_clusters) + ")"

# plot
sc.pl.embedding(
    adata_plot,
    basis=basis,
    color=color_by,
    s=point_size,
    frameon=False,
    palette=palette,
    vmin=vmin,
    vmax=vmax,
    legend_loc=legend_loc,
    title=title,
    show=False,
)

plt.gcf().set_size_inches(plot_width, plot_height)
plt.tight_layout()
plt.show()

### kNN neighborhood analysis

In [ ]:
### ECM neighborhood analysis (sample-balanced, self-aware, tunable)

# %%
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import gc

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from pandas.api.types import CategoricalDtype

# USER SETTINGS
cluster_key   = "ECM_leiden"        # categorical ECM clusters per spot
nhood_key     = "ECM_neighborhood"  # output column name for neighborhoods
coord_type    = "grid"              # Visium hex topology
n_rings       = 1                   # spatial radius in rings (1 = immediate neighbors)

# Smoothing of neighbor composition
alpha_global  = 0.1                 # shrink toward uniform composition
alpha_self    = 1.0                 # weight of the seed spot's own ECM cluster
pseudocount   = 1e-6                # floor to avoid zeros

apply_hellinger = True              # Hellinger transform of compositions

# Embedding
n_pca_components = 15               # PCA dimensions for neighborhood vectors

# k-means clustering
kmeans_n_clusters    = 10           # number of neighborhoods
kmeans_n_init        = 25
kmeans_random_state  = 42

# Sample-balanced training (optional for speed)
max_spots_per_sample_for_kmeans = None  # None to use all spots; otherwise cap per sample ex 2000

# Reporting thresholds
min_patients_report = 5             # neighborhoods present in fewer than this many patients are "rare"


# 0) CHECK INPUT

if "adata_target" in locals():
    adata = adata_target
elif "adata" in locals():
    adata = adata
else:
    raise ValueError("Expected 'adata_target' or 'adata' in memory.")

req_cols = ["sample", "imagecol", "imagerow", cluster_key]
missing = [c for c in req_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required obs columns: {missing}")

if not isinstance(adata.obs[cluster_key].dtype, CategoricalDtype):
    adata.obs[cluster_key] = adata.obs[cluster_key].astype("category")

cats = adata.obs[cluster_key].cat.categories
C = cats.size
if C < 2:
    raise ValueError(f"Need >= 2 ECM clusters in '{cluster_key}', found {C}.")

print(f"Using AnnData with {adata.n_obs} spots, {C} ECM clusters.")
print(f"Neighborhoods based on {coord_type} topology, n_rings = {n_rings}.")


# 1–5) PER-SAMPLE NEIGHBOR COMPOSITIONS (with self + global smoothing)

samples = adata.obs["sample"].unique().tolist()
all_feats = []
all_barcodes = []
all_samples = []

for sid in samples:
    ad_s = adata[adata.obs["sample"] == sid].copy()
    if ad_s.n_obs == 0:
        continue

    if "spatial" not in ad_s.obsm:
        ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()

    # Build spatial neighbors for this sample
    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr()

    # One-hot of ECM cluster labels (N x C)
    codes = ad_s.obs[cluster_key].cat.codes.to_numpy()
    X_one = np.eye(C, dtype=np.float32)[codes]

    # Neighbor composition (counts)
    X_nb = A.dot(X_one).astype(np.float32)

    # Smoothing: self + global + floor
    # self: add alpha_self to the seed's own cluster
    # global: alpha_global / C added to each cluster as weak prior
    X_nb += alpha_self * X_one
    X_nb += alpha_global * (1.0 / C)
    X_nb += pseudocount

    # Normalize to probabilities
    row_sums = X_nb.sum(axis=1, dtype=np.float32)
    X_nb /= row_sums[:, None]

    all_feats.append(X_nb)
    all_barcodes.extend(ad_s.obs_names.to_list())
    all_samples.extend(ad_s.obs["sample"].astype(str).to_list())

    # Cleanup
    del ad_s, A, X_one, X_nb, row_sums, codes
    gc.collect()

if not all_feats:
    raise RuntimeError("No neighborhood features were generated; check inputs and clusters.")


# 6) COMBINE, OPTIONAL HELLINGER, PREP FOR PCA

X = np.vstack(all_feats).astype(np.float32)   # (N_total x C)
del all_feats
gc.collect()

all_barcodes = np.array(all_barcodes)
all_samples_arr = np.array(all_samples)
del all_samples
gc.collect()

if apply_hellinger:
    np.sqrt(X, out=X)

print("Neighborhood feature matrix shape:", X.shape)


# 7) SAMPLE-BALANCED SUBSET FOR PCA + K-MEANS

rng = np.random.default_rng(kmeans_random_state)
unique_samples = np.unique(all_samples_arr)

train_indices = []

if max_spots_per_sample_for_kmeans is None:
    train_indices = np.arange(X.shape[0], dtype=int)
else:
    for sid in unique_samples:
        idx_s = np.where(all_samples_arr == sid)[0]
        if idx_s.size == 0:
            continue
        if idx_s.size > max_spots_per_sample_for_kmeans:
            idx_s = rng.choice(idx_s, size=max_spots_per_sample_for_kmeans, replace=False)
        train_indices.append(idx_s)
    if len(train_indices) == 0:
        raise RuntimeError("No training indices selected. Check sampling settings.")
    train_indices = np.concatenate(train_indices)

train_indices = np.sort(train_indices)
print(f"Training subset size for PCA + k-means: {train_indices.size} spots "
      f"({train_indices.size / X.shape[0]:.3%} of all spots).")

X_train = X[train_indices]


# 8) PCA ON TRAINING SUBSET, TRANSFORM ALL

p = min(n_pca_components, X.shape[1] - 1)
if p < 1:
    raise ValueError("Cannot compute PCA with fewer than 1 component.")

pca = PCA(n_components=p, svd_solver="randomized", random_state=kmeans_random_state)
X_train_pca = pca.fit_transform(X_train).astype(np.float32)
X_pca = pca.transform(X).astype(np.float32)

print(f"PCA components used: {p}")
print("Explained variance (first 5 PCs):", np.round(pca.explained_variance_ratio_[:5], 4))

del X_train
gc.collect()


# 9) K-MEANS ON TRAINING SUBSET, PREDICT ALL

km = KMeans(
    n_clusters=kmeans_n_clusters,
    n_init=kmeans_n_init,
    random_state=kmeans_random_state,
    verbose=0,
)

km.fit(X_train_pca)
labels_all = km.predict(X_pca).astype(np.int32)

del X_pca, X_train_pca
gc.collect()

print("k-means clustering complete.")
print("Neighborhood label range:", labels_all.min(), "to", labels_all.max())


# 10) MAP BACK TO FULL OBJECT

labels_ser = pd.Series(labels_all, index=pd.Index(all_barcodes, name="barcode"))
labels_ser = labels_ser.astype("int32").astype("string")
mapped = labels_ser.reindex(adata.obs_names)

adata.obs[nhood_key] = mapped.astype("category")

sizes = adata.obs[nhood_key].value_counts(dropna=True).sort_values(ascending=False)
print(f"Number of ECM neighborhoods: {sizes.size}")
print("\nTop neighborhood sizes:")
print(sizes.head(20))


# 11) PER-PATIENT COVERAGE SUMMARY

if "sample" not in adata.obs.columns:
    raise KeyError("adata.obs['sample'] is required for per-patient summaries.")

cluster_sizes = adata.obs[nhood_key].value_counts(dropna=True)
patient_counts = (
    adata.obs.dropna(subset=[nhood_key])
         .groupby(nhood_key)["sample"]
         .nunique()
)

summary = pd.DataFrame({
    "n_spots": cluster_sizes,
    "n_patients": patient_counts,
}).sort_values("n_spots", ascending=False)

print("\nNeighborhood per-patient coverage summary (head):")
print(summary.head(20))

rare = summary[summary["n_patients"] < min_patients_report]
print(f"\nNeighborhoods present in fewer than {min_patients_report} patients: {rare.shape[0]}")
if not rare.empty:
    print(rare)

### Visualize on a sample

In [ ]:
# Plot ECM neighborhoods on a single Visium sample

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import matplotlib.patches as mpatches

# choose AnnData
if "adata_target" in locals():
    adata = adata_target
elif "adata_subset" in locals():
    adata = adata_subset
elif "adata_cohort" in locals():
    adata = adata_cohort
elif "adata" in locals():
    adata = adata
else:
    raise ValueError("No AnnData object found.")

sample_key  = "sample"
cluster_key = "ECM_neighborhood"
sample_id   = "CH_G_196a_vis"   # change as needed

spot_size   = 80
alpha       = 0.9
edge_color  = None
cmap_name   = "tab20"

if sample_key not in adata.obs.columns:
    raise KeyError(f"{sample_key} not in adata.obs")
if cluster_key not in adata.obs.columns:
    raise KeyError(f"{cluster_key} not in adata.obs")
if sample_id not in adata.obs[sample_key].unique():
    raise ValueError(f"Sample '{sample_id}' not found in adata.obs['{sample_key}'].")

if adata.obs[cluster_key].dtype.name != "category":
    adata.obs[cluster_key] = adata.obs[cluster_key].astype("category")

cats = adata.obs[cluster_key].cat.categories.astype(str).tolist()

cats_key   = f"{cluster_key}_categories"
colors_key = f"{cluster_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cats_global   = list(adata.uns[cats_key])
    colors_global = list(adata.uns[colors_key])

    # rebuild if categories changed or lengths mismatch
    if (len(cats_global) != len(colors_global)) or (set(cats_global) != set(cats)):
        cmap = plt.get_cmap(cmap_name, len(cats))
        colors = [to_hex(cmap(i)) for i in range(len(cats))]
        adata.uns[cats_key]   = cats
        adata.uns[colors_key] = colors
        cm = dict(zip(cats, colors))
    else:
        cm = dict(zip(cats_global, colors_global))
else:
    cmap = plt.get_cmap(cmap_name, len(cats))
    colors = [to_hex(cmap(i)) for i in range(len(cats))]
    adata.uns[cats_key]   = cats
    adata.uns[colors_key] = colors
    cm = dict(zip(cats, colors))

ad = adata[adata.obs[sample_key] == sample_id].copy()
labels = ad.obs[cluster_key].astype(str)

if "spatial" in ad.obsm:
    xy = ad.obsm["spatial"]
else:
    if not {"imagecol", "imagerow"}.issubset(ad.obs.columns):
        raise KeyError("Need obsm['spatial'] or obs['imagecol','imagerow'].")
    xy = ad.obs[["imagecol", "imagerow"]].to_numpy()

point_colors = labels.map(lambda x: cm.get(x, "#999999")).to_numpy()

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(
    xy[:, 0],
    xy[:, 1],
    c=point_colors,
    s=spot_size,
    edgecolors="none" if edge_color is None else edge_color,
    linewidths=0 if edge_color is None else 0.1,
    alpha=alpha,
)

ax.invert_yaxis()
ax.axis("off")
ax.set_title(f"{sample_id} – {cluster_key}")

present = set(labels.unique().tolist())
handles = []
for k in cats:
    if (k in present) and (k in cm):
        handles.append(mpatches.Patch(color=cm[k], label=k))

if handles:
    ax.legend(
        handles=handles,
        loc="upper right",
        bbox_to_anchor=(1.3, 1),
        frameon=False,
        fontsize="small",
        title="Neighborhood",
    )

plt.tight_layout()
plt.show()

### Proportion across cohort

In [ ]:
# %% [markdown]
# ECM neighborhood proportions across patients (stacked bar)

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.colors import to_hex

# user params
nhood_key              = "ECM_neighborhood"
sample_key             = "sample"
normalize              = True          # per-patient proportions if True, counts if False
top_n_neighborhoods    = None          # e.g. 12; None = keep all
other_label            = "Other"
min_spots_per_sample   = 200
cluster_samples        = True
cluster_neighborhoods  = True
fig_width, fig_height  = 18, 8
legend_outside         = True
legend_ncol            = 2
cmap_name              = "tab20"

# pick AnnData
if "adata_target" in locals():
    adata = adata_target
elif "adata_subset" in locals():
    adata = adata_subset
elif "adata_cohort" in locals():
    adata = adata_cohort
elif "adata" in locals():
    adata = adata
else:
    raise ValueError("No AnnData object found.")

# basic checks
for c in (sample_key, nhood_key):
    if c not in adata.obs.columns:
        raise KeyError(f"{c} not in adata.obs")

if not adata.obs[nhood_key].dtype.name == "category":
    adata.obs[nhood_key] = adata.obs[nhood_key].astype("category")

# build sample x neighborhood matrix
df = adata.obs[[sample_key, nhood_key]].dropna()
pivot = df.groupby([sample_key, nhood_key]).size().unstack(fill_value=0)

# filter low-spot samples
pivot = pivot.loc[pivot.sum(axis=1) >= min_spots_per_sample]
if pivot.empty:
    raise ValueError("No samples left after min_spots_per_sample filter.")

# collapse to top N neighborhoods if requested
if top_n_neighborhoods is not None and pivot.shape[1] > top_n_neighborhoods:
    totals = pivot.sum(axis=0).sort_values(ascending=False)
    keep = totals.index[:top_n_neighborhoods]
    other = pivot.drop(columns=keep).sum(axis=1)
    pivot = pivot[keep]
    pivot[other_label] = other

# normalize per sample if desired
if normalize:
    pivot = pivot.div(pivot.sum(axis=1), axis=0)

# optional clustering of neighborhoods (columns)
if cluster_neighborhoods and pivot.shape[1] > 2:
    Zc = linkage(pivot.T.values.astype(np.float32), method="average", metric="cosine")
    order_cols = dendrogram(Zc, no_plot=True)["leaves"]
    pivot = pivot.iloc[:, order_cols]
else:
    pivot = pivot.loc[:, pivot.sum(0).sort_values(ascending=False).index]

# optional clustering of samples (rows)
if cluster_samples and pivot.shape[0] > 2:
    Zs = linkage(pivot.values.astype(np.float32), method="average", metric="cosine")
    order_rows = dendrogram(Zs, no_plot=True)["leaves"]
    pivot = pivot.iloc[order_rows]

# global color map (shared with other plots)
cats_key   = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cats_global   = list(adata.uns[cats_key])
    colors_global = list(adata.uns[colors_key])
    if len(cats_global) != len(colors_global):
        raise ValueError(f"Length mismatch for {cats_key} and {colors_key}.")
    cm = dict(zip(cats_global, colors_global))
else:
    cats_global = adata.obs[nhood_key].cat.categories.astype(str).tolist()
    cmap = plt.get_cmap(cmap_name, len(cats_global))
    colors_global = [to_hex(cmap(i)) for i in range(len(cats_global))]
    adata.uns[cats_key]   = cats_global
    adata.uns[colors_key] = colors_global
    cm = dict(zip(cats_global, colors_global))

# colors for columns (use gray for "Other" or unknown)
colors = []
for col in pivot.columns.astype(str):
    if col == other_label:
        colors.append("#b0b0b0")
    else:
        colors.append(cm.get(col, "#999999"))

# stacked bar plot
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
x = np.arange(pivot.shape[0])
bottom = np.zeros(pivot.shape[0], dtype=float)

for col, color in zip(pivot.columns, colors):
    vals = pivot[col].to_numpy()
    ax.bar(x, vals, bottom=bottom, width=0.9, color=color, edgecolor="none", label=str(col))
    bottom += vals

ax.set_xlim(-0.5, len(x) - 0.5)
ax.set_ylabel("Proportion" if normalize else "Spot count")
ax.set_title("ECM neighborhoods across patients")

step = max(1, len(x) // 40)
ax.set_xticks(x[::step])
ax.set_xticklabels(pivot.index[::step], rotation=90, fontsize=8)
ax.margins(x=0)
ax.grid(False)

if legend_outside:
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.0, 0.5),
        frameon=False,
        ncol=legend_ncol,
        fontsize=8,
    )
else:
    ax.legend(frameon=False, ncol=legend_ncol, fontsize=8)

plt.tight_layout()
plt.show()

pivot.to_csv("ECM_neighborhoods_by_patient_stacked_matrix.csv")
print(f"Saved matrix with shape {pivot.shape} to 'ECM_neighborhoods_by_patient_stacked_matrix.csv'")

### ECM neighborhood cell landscape

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.colors import to_hex

nhood_key    = "ECM_neighborhood"
c2l_prefix   = "meanscell_abundance_w_sf_"
curation_col = "celltype_level4_curated"

top_n        = 12
figsize      = (16, 6)
bar_width    = 0.75
other_color  = "#ebecf0"
legend_font  = 8
legend_ncol  = 1

# Exclude patterns (case-insensitive)
EXCLUDE_PATTERNS = ["doublet", "other"]

if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

c2l_cols = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])
counts = adata.obs[c2l_cols].copy()
counts.columns = lvl4_names

# Map to curated names
lvl4_in_key = lvl4_names.intersection(celltype_key.index)
counts = counts[lvl4_in_key]
curated_map = celltype_key.loc[lvl4_in_key, curation_col].astype(str).fillna("Unknown")

# Filter out excluded cell types before aggregation
keep_mask = ~curated_map.str.lower().apply(
    lambda x: any(pat in x for pat in EXCLUDE_PATTERNS)
)
counts = counts.loc[:, keep_mask.values]
curated_map = curated_map[keep_mask]

print(f"Excluded cell types: {(~keep_mask).sum()}")
print(f"Retained cell types: {keep_mask.sum()}")

counts_curated = counts.groupby(curated_map, axis=1).sum()

# Normalize to fractions
row_sums = counts_curated.sum(axis=1).replace(0, np.nan)
frac = counts_curated.div(row_sums, axis=0).fillna(0.0)

# Mean fraction per neighborhood, re-normalize
nh = adata.obs[nhood_key].astype("category")
means = (
    frac.groupby(nh)
        .mean()
        .pipe(lambda df: df.div(df.sum(axis=1), axis=0))
        .fillna(0.0)
)

# Order neighborhoods by hierarchical clustering
H = np.sqrt(means.values.astype(np.float64))
Z = linkage(H, method="average", metric="euclidean")
order_idx = leaves_list(Z)
nhood_order = [means.index[i] for i in order_idx]

# Get top N cell types per neighborhood
top_map = {}
for nh_name, row in means.iterrows():
    top_map[nh_name] = row.sort_values(ascending=False).head(top_n)

# Union of all top types, sorted by global importance
union_types = sorted(set().union(*[s.index for s in top_map.values()]))
global_importance = pd.Series(0.0, index=union_types)
for s in top_map.values():
    global_importance.loc[s.index] += s.values
union_types_sorted = list(global_importance.sort_values(ascending=False).index)

# Color palette
tab20 = list(plt.get_cmap("tab20").colors) + list(plt.get_cmap("tab20b").colors)
palette = {ct: to_hex(tab20[i % len(tab20)]) for i, ct in enumerate(union_types_sorted)}

# Build stacked bar data
x = np.arange(len(nhood_order))
heights_by_type = {
    ct: np.array([float(top_map[nh].get(ct, 0.0)) for nh in nhood_order])
    for ct in union_types_sorted
}
other_heights = np.array(
    [max(0.0, 1.0 - float(top_map[nh].sum())) for nh in nhood_order]
)

# Plot
fig, ax = plt.subplots(figsize=figsize)
bottom = np.zeros(len(nhood_order), dtype=float)

for ct in union_types_sorted:
    h = heights_by_type[ct]
    if np.all(h == 0):
        continue
    ax.bar(x, h, bottom=bottom, width=bar_width, label=ct,
           color=palette[ct], edgecolor="white", linewidth=0.3)
    bottom += h

if not np.all(other_heights == 0):
    ax.bar(x, other_heights, bottom=bottom, width=bar_width, label="Other",
           color=other_color, edgecolor="white", linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels([f"N{nh}" for nh in nhood_order], fontsize=11, fontweight="bold")
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_xlim(-0.5, len(x) - 0.5)
ax.margins(x=0.01)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_title("Cell-type composition per ECM neighborhood", fontsize=13, fontweight="bold")

# Legend outside right, sorted by importance
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels,
          loc="center left", bbox_to_anchor=(1.01, 0.5),
          frameon=False, fontsize=legend_font, ncol=legend_ncol,
          title="Cell type", title_fontsize=9, handlelength=1.2,
          handleheight=1.0, labelspacing=0.4)

plt.tight_layout()
plt.savefig("ECM_neighborhood_cell_landscape.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: ECM_neighborhood_cell_landscape.png")

### Correlate ECM neighborhoods with survival

In [ ]:
# Build clin_dedup with join_key (one row per patient, prefer 'a' block)
clin_out["block_suffix"] = clin_out["Block sequence number per subject for MOSAIC"].astype(str)
clin_out["join_key"] = clin_out["Subject Id"].astype(str) + clin_out["block_suffix"]

suffix_order = {"a": 0, "b": 1, "c": 2}
clin_out["_sort"] = clin_out["block_suffix"].map(suffix_order).fillna(99)
clin_dedup = (
    clin_out.sort_values("_sort")
    .drop_duplicates(subset=["Subject Id"], keep="first")
    .drop(columns=["_sort"])
    .copy()
)
print(f"clin_dedup: {len(clin_dedup)} patients, {clin_dedup['join_key'].nunique()} unique join keys")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from matplotlib.colors import to_hex

nhood_key  = "ECM_neighborhood"
sample_key = "sample"
min_spots  = 100
top_labels = 10
cmap_name  = "tab20"

# Basic checks
for c in (nhood_key, sample_key):
    if c not in adata.obs.columns:
        raise KeyError(f"{c} not found in adata.obs")

sdf = adata.obs[[sample_key, nhood_key]].dropna().copy()
sdf[nhood_key] = sdf[nhood_key].astype("category")
if sdf[nhood_key].cat.categories.size == 0:
    raise ValueError("No neighborhoods present")

# Per-sample neighborhood fractions
counts = sdf.groupby([sample_key, nhood_key]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1) >= min_spots]
if counts.empty:
    raise ValueError("No samples pass min_spots filter")

fracs = counts.div(counts.sum(axis=1), axis=0)

# Link to clinical via join_key
map_df = pd.DataFrame({sample_key: fracs.index.astype(str)})
map_df["join_key"] = map_df[sample_key].str.replace(r"_vis$", "", regex=True).str[:9]

clin_os = clin_dedup[["join_key", "os_years"]].dropna(subset=["os_years"]).copy()

M = (
    fracs.reset_index()
         .merge(map_df, on=sample_key, how="left")
         .merge(clin_os, on="join_key", how="inner")
         .dropna(subset=["os_years"])
)

if M.empty:
    raise ValueError("No overlap between neighborhood fractions and clinical data")

print(f"Visium samples with neighborhoods: {fracs.shape[0]}")
print(f"Samples with OS after merge:       {M[sample_key].nunique()}")

# Spearman correlation per neighborhood
res = []
y = M["os_years"].to_numpy()
for c in fracs.columns:
    x = M[c].to_numpy()
    r, p = spearmanr(x, y)
    if np.isfinite(r) and np.isfinite(p):
        res.append({"neighborhood": str(c), "rho": r, "pval": p})

if not res:
    raise ValueError("No valid Spearman correlations")

df_res = pd.DataFrame(res)

# BH FDR
p = df_res["pval"].values
n = len(p)
order = np.argsort(p)
rank = np.arange(1, n + 1, dtype=float)
fdr = np.empty(n)
fdr[order] = np.minimum.accumulate((p[order] * n / rank)[::-1])[::-1]
df_res["fdr"] = np.clip(fdr, 0, 1)
df_res["neglog10p"] = -np.log10(np.clip(df_res["pval"].values, np.finfo(float).tiny, 1))
df_res = df_res.sort_values("pval").reset_index(drop=True)

# Color map consistent with other plots
cats = adata.obs[nhood_key].astype("category").cat.categories.astype(str).tolist()
cats_key = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cm = dict(zip(adata.uns[cats_key], adata.uns[colors_key]))
else:
    cmap = plt.get_cmap(cmap_name, len(cats))
    colors = [to_hex(cmap(i)) for i in range(len(cats))]
    adata.uns[cats_key] = cats
    adata.uns[colors_key] = colors
    cm = dict(zip(cats, colors))

# Volcano plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    df_res["rho"], df_res["neglog10p"], s=90,
    c=[cm.get(k, "#666666") for k in df_res["neighborhood"]],
    edgecolors="white", linewidths=0.5,
)
ax.axvline(0, ls="--", c="grey", lw=1)
ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=1)

for i in range(min(top_labels, len(df_res))):
    ax.annotate(
        df_res.at[i, "neighborhood"],
        (df_res.at[i, "rho"], df_res.at[i, "neglog10p"]),
        xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=10,
    )

ax.set(xlabel="Spearman ρ (neighborhood burden vs OS)", ylabel="-log10(p)",
       title="ECM neighborhood abundance vs overall survival")
ax.grid(False)
plt.tight_layout()
plt.show()

# Print and save
print(df_res.to_string(index=False))
df_res.to_csv("ECM_neighborhood_vs_OS_spearman.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
import os

ro.r("suppressPackageStartupMessages(library(survRM2))")

# =============================================================================
# CONFIG
# =============================================================================

nhood_key  = "ECM_neighborhood"
min_spots  = 10
tau_years  = 3
endpoint   = "OS"
cmap_name  = "tab20"

SPLIT_MODES = ["median", "quartile", "max"]

# Toggle covariates on/off
USE_MGMT = True
USE_AGE  = False
USE_SEX  = False
USE_P53  = False

FOREST_SPLIT = "median"  # update after reviewing sweep summary

SAVE_DIR     = "/home/sagemaker-user/gbm-collaboration_2/analysis_ecm"
FP_SAVE_STEM = f"{SAVE_DIR}/forest_rmst_ecm_neighborhoods"
os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# AESTHETICS
# =============================================================================

FP_FIG_W     = 9
FP_FIG_H     = 6.0
FP_DPI       = 300
FP_FS_TITLE  = 9
FP_FS_AXIS   = 12
FP_FS_HDR    = 8
FP_FS_ROW    = 8
FP_MARKER    = "D"
FP_MARKER_SZ = 5
FP_CAP_SZ    = 3
FP_LW        = 3
FP_ELW       = 2
FP_COL_NS    = "#888888"
FP_SIG_ALPHA = 0.05
X_LABEL      = -0.02
X_EST        =  1.02
X_P          =  1.28

# =============================================================================
# COVARIATE DEFINITIONS
# =============================================================================

COVARIATE_DEFS = {
    "mgmt": {
        "col":       "MGMT promoter methylation",
        "transform": lambda x: (x == "Methylated").astype(float),
        "active":    USE_MGMT,
    },
    "age": {
        "col":       "age_at_diagnosis",
        "transform": lambda x: x.astype(float),
        "active":    USE_AGE,
    },
    "sex": {
        "col":       "Administrative gender",
        "transform": lambda x: (x == "Female").astype(float),
        "active":    USE_SEX,
    },
    "p53": {
        "col":       "p53",
        "transform": lambda x: (x == "Positive").astype(float),
        "active":    USE_P53,
    },
}

COVARIATES = {k: v for k, v in COVARIATE_DEFS.items() if v["active"]}
cov_names  = list(COVARIATES.keys())
cov_label  = " + ".join(cov_names) if cov_names else "none"
adjust     = len(cov_names) > 0

print(f"Active covariates: {cov_label}")

# =============================================================================
# ENDPOINT
# =============================================================================

if endpoint.upper() == "OS":
    time_col, event_col = "os_years", "os_event"
    time_label = "Overall Survival"
else:
    time_col, event_col = "pfs_years", "pfs_event"
    time_label = "Progression-Free Survival"

# =============================================================================
# NEIGHBORHOOD FRACTIONS
# =============================================================================

sdf    = adata.obs[["sample", nhood_key]].dropna().copy()
sdf[nhood_key] = sdf[nhood_key].astype("category")
counts = sdf.groupby(["sample", nhood_key], observed=False).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1) >= min_spots]
fracs  = counts.div(counts.sum(axis=1), axis=0)
print(f"Samples with >= {min_spots} spots: {len(fracs)}")

# =============================================================================
# CLINICAL DATA
# =============================================================================

all_cov_cols = [v["col"] for v in COVARIATES.values()]
clin_cols    = ["join_key", time_col, event_col] + all_cov_cols
clin_os      = clin_dedup[clin_cols].dropna(subset=[time_col, event_col]).copy()

for cov_name, cov_spec in COVARIATES.items():
    clin_os[cov_name] = cov_spec["transform"](clin_os[cov_spec["col"]])

if cov_names:
    clin_os = clin_os.dropna(subset=cov_names)

map_df             = pd.DataFrame({"sample": fracs.index.astype(str)})
map_df["join_key"] = map_df["sample"].str.replace(r"_vis$", "", regex=True).str[:9]

merge_cols = ["join_key", time_col, event_col] + cov_names
M = (
    fracs.reset_index()
         .merge(map_df, on="sample", how="left")
         .merge(clin_os[merge_cols], on="join_key", how="inner")
)
print(f"Merged patients: {M['sample'].nunique()}")

# =============================================================================
# STRATIFICATION HELPER
# =============================================================================

def apply_split(series, mode):
    if mode == "median":
        thr  = series.median()
        arm  = (series >= thr).astype(int)
        keep = pd.Series(True, index=series.index)
    elif mode == "quartile":
        q25, q75 = series.quantile([0.25, 0.75])
        arm  = (series >= q75).astype(int)
        keep = (series >= q75) | (series <= q25)
    elif mode == "max":
        q75  = series.quantile(0.75)
        arm  = (series >= q75).astype(int)
        keep = pd.Series(True, index=series.index)
    else:
        raise ValueError(f"Unknown split mode: {mode}")
    return arm, keep

# =============================================================================
# RMST HELPER
# =============================================================================

def run_rmst(df_in, nh_col, tau):
    df = df_in[[time_col, event_col, "arm", nh_col] + cov_names].dropna().copy()
    df = df[df[time_col] > 0]

    if len(df) == 0 or df[time_col].max() < tau:
        return None

    n_high = int(df["arm"].sum())
    n_low  = int((df["arm"] == 0).sum())
    if n_high < 10 or n_low < 10:
        return None

    r_df = df[[time_col, event_col, "arm"] + cov_names].copy()
    r_df.columns = ["time", "event", "arm"] + cov_names

    try:
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
        ro.globalenv["tau_val"] = float(tau)

        ro.r("""
        rmst_u <- rmst2(surv_data$time, surv_data$event, surv_data$arm, tau=tau_val)
        u_diff <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "Est."]
        u_lo   <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "lower .95"]
        u_hi   <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "upper .95"]
        u_p    <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "p"]
        rmst_h <- rmst_u$RMST.arm1$rmst["Est."]
        rmst_l <- rmst_u$RMST.arm0$rmst["Est."]
        """)

        out = {
            "n_high":    n_high,
            "n_low":     n_low,
            "rmst_high": float(ro.r("rmst_h")[0]),
            "rmst_low":  float(ro.r("rmst_l")[0]),
            "none_diff": float(ro.r("u_diff")[0]),
            "none_lo":   float(ro.r("u_lo")[0]),
            "none_hi":   float(ro.r("u_hi")[0]),
            "none_p":    float(ro.r("u_p")[0]),
        }

        if adjust:
            cov_parts = []
            for c in cov_names:
                if c == "age":
                    cov_parts.append(f"scale(surv_data${c})[,1]")
                else:
                    cov_parts.append(f"surv_data${c}")
            cov_r_str = ", ".join(cov_parts)

            ro.r(f"""
            cov_mat <- cbind({cov_r_str})
            rmst_a <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                            tau=tau_val, covariates=cov_mat)
            a_diff <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "Est."]
            a_lo   <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "lower .95"]
            a_hi   <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "upper .95"]
            a_p    <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "p"]
            """)
            out.update({
                "active_diff": float(ro.r("a_diff")[0]),
                "active_lo":   float(ro.r("a_lo")[0]),
                "active_hi":   float(ro.r("a_hi")[0]),
                "active_p":    float(ro.r("a_p")[0]),
            })
        else:
            out.update({
                "active_diff": out["none_diff"],
                "active_lo":   out["none_lo"],
                "active_hi":   out["none_hi"],
                "active_p":    out["none_p"],
            })

        return out

    except Exception as e:
        print(f"    survRM2 error: {e}")
        return None

# =============================================================================
# SWEEP
# =============================================================================

all_results = []

for split_mode in SPLIT_MODES:
    print(f"\n--- Split mode: {split_mode} ---")
    for nh_col in fracs.columns:
        df = M[[time_col, event_col, nh_col] + cov_names].dropna().copy()
        df = df[df[time_col] > 0].copy()

        arm, keep = apply_split(df[nh_col], split_mode)
        df["arm"] = arm
        df = df[keep].copy()

        res = run_rmst(df, nh_col, tau_years)
        if res:
            res["neighborhood"] = str(nh_col)
            res["split_mode"]   = split_mode
            all_results.append(res)
        else:
            print(f"  {nh_col}: skipped")

sweep_df = pd.DataFrame(all_results)

# =============================================================================
# SWEEP SUMMARY — nominal p-values, no FDR correction
# =============================================================================

print("\n" + "="*60)
print(f"SWEEP SUMMARY  |  covariates: {cov_label}  |  tau={tau_years}y")
print("="*60)

for mode in SPLIT_MODES:
    sub = sweep_df[sweep_df["split_mode"] == mode].copy()
    if len(sub) == 0:
        continue
    n_sig = (sub["active_p"] < 0.05).sum()
    best_p = sub["active_p"].min()
    print(f"\n  {mode}: {n_sig} sig (p<0.05), best p={best_p:.4f}")
    print(sub[["neighborhood", "n_high", "n_low", "active_diff", "active_p"]]
          .assign(active_diff=lambda d: (d["active_diff"] * 12).round(2))
          .sort_values("active_p")
          .to_string(index=False))

# =============================================================================
# FOREST PLOT — nominal p-values
# Set FOREST_SPLIT above to preferred mode, then re-run
# =============================================================================

sub     = sweep_df[sweep_df["split_mode"] == FOREST_SPLIT].copy()
sub     = sub.sort_values("active_p")

# Neighborhood colors from adata
cats = adata.obs[nhood_key].astype("category").cat.categories.astype(str).tolist()
if f"{nhood_key}_colors" in adata.uns:
    cm = dict(zip(
        adata.uns.get(f"{nhood_key}_categories", cats),
        adata.uns[f"{nhood_key}_colors"]
    ))
else:
    cmap_obj = plt.get_cmap(cmap_name, len(cats))
    cm = {c: to_hex(cmap_obj(i)) for i, c in enumerate(cats)}

def fmt_p(p):
    if np.isnan(p):
        return "—"
    if p < 0.0001:
        return f"{p:.2e}"
    return f"{p:.4f}"

rows = []
for _, row in sub.iterrows():
    d   = row["active_diff"] * 12
    lo  = row["active_lo"]   * 12
    hi  = row["active_hi"]   * 12
    p   = row["active_p"]
    sig = p < FP_SIG_ALPHA
    rows.append({
        "label": str(row["neighborhood"]),
        "diff":  d,
        "lo":    lo,
        "hi":    hi,
        "p":     p,
        "sig":   sig,
        "dir":   "pos" if d >= 0 else "neg",
        "color": cm.get(str(row["neighborhood"]), "#666666"),
    })

rows_pos = sorted([r for r in rows if r["dir"] == "pos"],
                   key=lambda x: x["diff"], reverse=True)
rows_neg = sorted([r for r in rows if r["dir"] == "neg"],
                   key=lambda x: x["diff"], reverse=True)
fp_rows  = rows_pos + rows_neg
n_rows   = len(fp_rows)
y_pos    = np.arange(n_rows - 1, -1, -1, dtype=float)

fig, ax = plt.subplots(figsize=(FP_FIG_W, FP_FIG_H))

for i, row in enumerate(fp_rows):
    y   = y_pos[i]
    col = row["color"]
    ax.errorbar(
        x=row["diff"], y=y,
        xerr=[[row["diff"] - row["lo"]], [row["hi"] - row["diff"]]],
        fmt=FP_MARKER, color=col,
        markersize=FP_MARKER_SZ, capsize=FP_CAP_SZ,
        linewidth=FP_LW, elinewidth=FP_ELW, zorder=3,
    )

ax.axvline(x=0, color="#7F8C8D", linewidth=0.9, linestyle="--", zorder=2)

all_lo = [r["lo"] for r in fp_rows]
all_hi = [r["hi"] for r in fp_rows]
x_pad  = max((max(all_hi) - min(all_lo)) * 0.08, 0.5)
ax.set_xlim(min(all_lo) - x_pad, max(all_hi) + x_pad)
ax.set_ylim(-0.7, n_rows - 0.3)
ax.set_yticks([])
ax.set_xlabel(
    f"RMST difference (months, τ={tau_years}y)  |  high vs low neighborhood fraction",
    fontsize=FP_FS_AXIS)
ax.set_title(
    f"ECM neighborhood survival associations "
    f"(τ={tau_years}y, {FOREST_SPLIT} split)\n"
    f"Covariates: {cov_label if cov_label else 'none'}  |  "
    f"colored = p<0.05, gray = n.s.",
    fontsize=FP_FS_TITLE, pad=8)
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_linewidth(0.6)
ax.tick_params(axis="x", labelsize=FP_FS_AXIS, width=0.6, length=3)
ax.grid(axis="x", linewidth=0.3, alpha=0.4, zorder=0)

trans      = ax.transAxes
y_lo, y_hi = ax.get_ylim()

def y_frac(y_data):
    return (y_data - y_lo) / (y_hi - y_lo)

header_frac = y_frac(n_rows - 0.1)
ax.text(X_LABEL, header_frac, "Neighborhood",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="right", va="bottom")
ax.text(X_EST, header_frac, "RMST diff (months)",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")
ax.text(X_P, header_frac, "adj p",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")

for i, row in enumerate(fp_rows):
    yf  = y_frac(y_pos[i])
    col = row["color"] if row["sig"] else FP_COL_NS
    ax.text(X_LABEL, yf, row["label"],
            transform=trans, fontsize=FP_FS_ROW,
            ha="right", va="center", color="#2C3E50")
    ax.text(X_EST, yf, f"{row['diff']:+.1f}",
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)
    ax.text(X_P, yf, fmt_p(row["p"]) + (" *" if row["sig"] else ""),
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)

fig.subplots_adjust(left=0.28, right=0.78)
fig.savefig(f"{FP_SAVE_STEM}_{FOREST_SPLIT}.png", dpi=FP_DPI, bbox_inches="tight")
fig.savefig(f"{FP_SAVE_STEM}_{FOREST_SPLIT}.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved {FP_SAVE_STEM}_{FOREST_SPLIT}.png/.pdf")

sweep_df.to_csv(f"{SAVE_DIR}/ecm_neighborhood_rmst_sweep.csv", index=False)
print(f"Saved sweep: {SAVE_DIR}/ecm_neighborhood_rmst_sweep.csv")

In [ ]:
def run_rmst(df_in, nh_col, tau):
    df = df_in[[time_col, event_col, "arm", nh_col] + cov_names].dropna().copy()
    df = df[df[time_col] > 0]

    if len(df) == 0 or df[time_col].max() < tau:
        return None

    n_high = int(df["arm"].sum())
    n_low  = int((df["arm"] == 0).sum())
    if n_high < 10 or n_low < 10:
        return None

    r_df = df[[time_col, event_col, "arm"] + cov_names].copy()
    r_df.columns = ["time", "event", "arm"] + cov_names

    try:
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
        ro.globalenv["tau_val"] = float(tau)

        # Unadjusted
        ro.r("""
        rmst_u <- rmst2(surv_data$time, surv_data$event, surv_data$arm, tau=tau_val)
        u_diff <- rmst_u$unadjusted.result[1, "Est."]
        u_lo   <- rmst_u$unadjusted.result[1, "lower .95"]
        u_hi   <- rmst_u$unadjusted.result[1, "upper .95"]
        u_p    <- rmst_u$unadjusted.result[1, "p"]
        rmst_h <- rmst_u$RMST.arm1$rmst["Est."]
        rmst_l <- rmst_u$RMST.arm0$rmst["Est."]
        """)

        out = {
            "n_high":    n_high,
            "n_low":     n_low,
            "rmst_high": float(ro.r("rmst_h")[0]),
            "rmst_low":  float(ro.r("rmst_l")[0]),
            "none_diff": float(ro.r("u_diff")[0]),
            "none_lo":   float(ro.r("u_lo")[0]),
            "none_hi":   float(ro.r("u_hi")[0]),
            "none_p":    float(ro.r("u_p")[0]),
        }

        if adjust:
            # Build covariate matrix in R exactly as old code does
            # Scale age if present, leave binary covariates as-is
            cov_parts = []
            for c in cov_names:
                if c == "age":
                    cov_parts.append(f"scale(surv_data${c})[,1]")
                else:
                    cov_parts.append(f"surv_data${c}")
            cov_r_str = ", ".join(cov_parts)

            ro.r(f"""
            cov_mat <- cbind({cov_r_str})
            rmst_a <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                            tau=tau_val, covariates=cov_mat)
            a_diff <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "Est."]
            a_lo   <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "lower .95"]
            a_hi   <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "upper .95"]
            a_p    <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "p"]
            """)
            out.update({
                "active_diff": float(ro.r("a_diff")[0]),
                "active_lo":   float(ro.r("a_lo")[0]),
                "active_hi":   float(ro.r("a_hi")[0]),
                "active_p":    float(ro.r("a_p")[0]),
            })
        else:
            out.update({
                "active_diff": out["none_diff"],
                "active_lo":   out["none_lo"],
                "active_hi":   out["none_hi"],
                "active_p":    out["none_p"],
            })

        return out

    except Exception as e:
        print(f"    survRM2 error: {e}")
        return None

In [ ]:
# Run the old single-neighborhood code with these exact settings
# and compare to the sweep output

chosen_neighborhood = 3
endpoint            = "OS"
split_mode          = "median"
tau_years           = 3
min_spots_per_samp  = 10

COVARIATES = {
    "mgmt": {
        "col": "MGMT promoter methylation",
        "transform": lambda x: (x == "Methylated").astype(float),
    },
    "age": {
        "col": "age_at_diagnosis",
        "transform": lambda x: x.astype(float),
    }
}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(survRM2))")

# ===== USER CONFIG =====
nhood_key    = "ECM_neighborhood"
min_spots    = 10
tau_years    = 3
split_mode   = "median"    # "median" or "quartile"
top_labels   = 10
cmap_name    = "tab20"
endpoint     = "OS"        # "OS" or "PFS"

# Covariates to adjust for (set to empty dict {} for unadjusted)
# Keys = name used in model, values = dict with col/transform info
COVARIATES = {
   "mgmt": {
        "col": "MGMT promoter methylation",
        "transform": lambda x: (x == "Methylated").astype(float),
    },
    "age": {
        "col": "age_at_diagnosis",
        "transform": lambda x: x.astype(float),
    }
}
# ========================

# Endpoint mapping
if endpoint.upper() == "OS":
    time_col, event_col = "os_years", "os_event"
    time_label = "Overall Survival"
elif endpoint.upper() == "PFS":
    time_col, event_col = "pfs_years", "pfs_event"
    time_label = "Progression-Free Survival"
else:
    raise ValueError("endpoint must be 'OS' or 'PFS'")

# Per-sample neighborhood fractions
sdf = adata.obs[["sample", nhood_key]].dropna()
counts = sdf.groupby(["sample", nhood_key]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1) >= min_spots]
fracs = counts.div(counts.sum(axis=1), axis=0)

# Clinical linkage + covariate prep
clin_cols = ["join_key", time_col, event_col] + [v["col"] for v in COVARIATES.values()]
clin_os = clin_dedup[clin_cols].dropna(subset=[time_col, event_col]).copy()

for cov_name, cov_spec in COVARIATES.items():
    clin_os[cov_name] = cov_spec["transform"](clin_os[cov_spec["col"]])

clin_os = clin_os.dropna(subset=list(COVARIATES.keys()))

map_df = pd.DataFrame({"sample": fracs.index.astype(str)})
map_df["join_key"] = map_df["sample"].str.replace(r"_vis$", "", regex=True).str[:9]

merge_cols = ["join_key", time_col, event_col] + list(COVARIATES.keys())
M = (
    fracs.reset_index()
         .merge(map_df, on="sample", how="left")
         .merge(clin_os[merge_cols], on="join_key", how="inner")
)

cov_names = list(COVARIATES.keys())
adjust = len(cov_names) > 0
cov_label = " + ".join(cov_names) if adjust else "none"

print(f"Samples: {M['sample'].nunique()}, tau={tau_years}, endpoint={endpoint}")
print(f"Covariates: {cov_label}")

# Screen all neighborhoods
results = []
for nh_col in fracs.columns:
    df = M[[time_col, event_col] + cov_names + [nh_col]].dropna().copy()
    df = df[df[time_col] > 0]

    # Split
    if split_mode == "median":
        thr = df[nh_col].median()
        df["arm"] = (df[nh_col] >= thr).astype(int)
    elif split_mode == "quartile":
        q25, q75 = df[nh_col].quantile([0.25, 0.75])
        df = df[(df[nh_col] <= q25) | (df[nh_col] >= q75)].copy()
        df["arm"] = (df[nh_col] >= q75).astype(int)
    else:
        raise ValueError("split_mode must be 'median' or 'quartile'")

    n_high = df["arm"].sum()
    n_low = len(df) - n_high

    if n_high < 10 or n_low < 10:
        continue
    if df[time_col].max() < tau_years:
        continue

    r_df = df[[time_col, event_col, "arm"] + cov_names].copy()
    r_df.columns = ["time", "event", "arm"] + cov_names

    try:
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
        ro.globalenv["tau"] = tau_years

        # Always run unadjusted
        ro.r("""
        rmst_unadj <- rmst2(time=surv_data$time, status=surv_data$event,
                            arm=surv_data$arm, tau=tau)
        """)

        unadj = ro.r("rmst_unadj$unadjusted.result[1,]")
        rmst_h = ro.r("rmst_unadj$RMST.arm1$rmst['Est.']")[0]
        rmst_l = ro.r("rmst_unadj$RMST.arm0$rmst['Est.']")[0]

        if adjust:
            cov_r_str = ", ".join([f'"{c}"' for c in cov_names])
            ro.r(f"""
            cov_mat <- as.matrix(surv_data[, c({cov_r_str})])
            rmst_adj <- rmst2(time=surv_data$time, status=surv_data$event,
                              arm=surv_data$arm, tau=tau, covariates=cov_mat)
            """)
            adj = ro.r("rmst_adj$adjusted.result[1,]")
            p_final = adj[3]
            diff_final = adj[0]
            ci_lo = adj[1]
            ci_hi = adj[2]
        else:
            p_final = unadj[3]
            diff_final = unadj[0]
            ci_lo = unadj[1]
            ci_hi = unadj[2]

        results.append({
            "neighborhood": str(nh_col),
            "n_high": n_high,
            "n_low": n_low,
            "rmst_high": rmst_h,
            "rmst_low": rmst_l,
            "diff": diff_final,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "p": p_final,
            "diff_unadj": unadj[0],
            "p_unadj": unadj[3],
        })
    except Exception as e:
        print(f"  Neighborhood {nh_col} failed: {e}")
        continue

df_rmst = pd.DataFrame(results).sort_values("p")
df_rmst["diff_months"] = df_rmst["diff"] * 12
df_rmst["neglog10p"] = -np.log10(np.clip(df_rmst["p"].values, np.finfo(float).tiny, 1))

print(f"\nScreened {len(df_rmst)} neighborhoods (tau={tau_years}, covariates: {cov_label})")
print(df_rmst[["neighborhood", "n_high", "n_low", "rmst_high", "rmst_low",
               "diff_months", "p", "p_unadj"]].round(4).to_string(index=False))

# Color map
cats = adata.obs[nhood_key].astype("category").cat.categories.astype(str).tolist()
cats_key = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"
if cats_key in adata.uns and colors_key in adata.uns:
    cm = dict(zip(adata.uns[cats_key], adata.uns[colors_key]))
else:
    cmap_obj = plt.get_cmap(cmap_name, len(cats))
    colors = [to_hex(cmap_obj(i)) for i in range(len(cats))]
    adata.uns[cats_key] = cats
    adata.uns[colors_key] = colors
    cm = dict(zip(cats, colors))

# Volcano plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_rmst["diff_months"], df_rmst["neglog10p"], s=100,
           c=[cm.get(k, "#666666") for k in df_rmst["neighborhood"]],
           edgecolors="white", linewidths=0.5, zorder=3)

ax.axvline(0, ls="--", c="grey", lw=1, alpha=0.5)
ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=1, alpha=0.5)

for i in range(min(top_labels, len(df_rmst))):
    row = df_rmst.iloc[i]
    ax.annotate(row["neighborhood"], (row["diff_months"], row["neglog10p"]),
                xytext=(0, 8), textcoords="offset points", ha="center", va="bottom", fontsize=9)

adj_str = f", {cov_label} adjusted" if adjust else ""
ax.set(xlabel=f"ΔRMST (months, High − Low{adj_str})",
       ylabel="-log10(p)",
       title=f"ECM neighborhood burden vs {time_label} (RMST, τ={tau_years} yr)")
ax.grid(False)
plt.tight_layout()
plt.show()

df_rmst.to_csv(f"ECM_neighborhood_vs_{endpoint}_rmst.csv", index=False)
print(f"Saved: ECM_neighborhood_vs_{endpoint}_rmst.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(survRM2))")

# ===== USER CONFIG =====
nhood_key           = "ECM_neighborhood"
chosen_neighborhood = 3
endpoint            = "OS"           # "OS" or "PFS"
split_mode          = "median"       # "median" or "quartile"
tau_years           = 3
min_spots_per_samp  = 100
min_group_n         = 5
km_alpha            = 0.10           # 90% CI

COL_HIGH = "#D55E00"
COL_LOW  = "#0072B2"

COVARIATES = {
    "mgmt": {
        "col": "MGMT promoter methylation",
        "transform": lambda x: (x == "Methylated").astype(float),
    },
    "age": {
        "col": "age_at_diagnosis",
        "transform": lambda x: x.astype(float),
    }
}
# ========================

# Endpoint mapping
if endpoint.upper() == "OS":
    time_col, event_col = "os_years", "os_event"
    time_label = "Overall survival (years)"
elif endpoint.upper() == "PFS":
    time_col, event_col = "pfs_years", "pfs_event"
    time_label = "Progression-free survival (years)"
else:
    raise ValueError("endpoint must be 'OS' or 'PFS'")

# Per-sample neighborhood fractions
sdf = adata.obs[["sample", nhood_key]].dropna().copy()
sdf[nhood_key] = sdf[nhood_key].astype("category")
counts = sdf.groupby(["sample", nhood_key]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1) >= min_spots_per_samp]
fracs = counts.div(counts.sum(axis=1), axis=0)

# Resolve neighborhood column
def resolve_nhood(cols, chosen):
    if chosen in cols:
        return chosen
    chosen_str = str(chosen)
    idx = np.where(cols.astype(str) == chosen_str)[0]
    if idx.size:
        return cols[idx[0]]
    raise ValueError(f"Neighborhood '{chosen}' not found. Available: {list(cols)}")

nh_col = resolve_nhood(fracs.columns, chosen_neighborhood)

# Clinical linkage with covariates
clin_cols = ["join_key", time_col, event_col] + [v["col"] for v in COVARIATES.values()]
clin_os = clin_dedup[clin_cols].dropna(subset=[time_col, event_col]).copy()
for cov_name, cov_spec in COVARIATES.items():
    clin_os[cov_name] = cov_spec["transform"](clin_os[cov_spec["col"]])
clin_os = clin_os.dropna(subset=list(COVARIATES.keys()))

map_df = pd.DataFrame({"sample": fracs.index.astype(str)})
map_df["join_key"] = map_df["sample"].str.replace(r"_vis$", "", regex=True).str[:9]

cov_names = list(COVARIATES.keys())
merge_cols = ["join_key", time_col, event_col] + cov_names

df = (
    fracs[[nh_col]].rename(columns={nh_col: "fraction"})
    .reset_index()
    .merge(map_df, on="sample", how="left")
    .merge(clin_os[merge_cols], on="join_key", how="inner")
    .dropna(subset=["fraction", time_col, event_col])
)
df["_time"] = df[time_col].astype(float)
df["_event"] = df[event_col].astype(int)
df = df[df["_time"] > 0]

# Split
if split_mode == "median":
    thr = df["fraction"].median()
    df["group"] = np.where(df["fraction"] >= thr, "High", "Low")
    split_note = f"median split ({thr:.3f})"
elif split_mode == "quartile":
    q25, q75 = df["fraction"].quantile([0.25, 0.75])
    df = df[(df["fraction"] <= q25) | (df["fraction"] >= q75)].copy()
    df["group"] = np.where(df["fraction"] >= q75, "High", "Low")
    split_note = f"Q1/Q4 split"
else:
    raise ValueError("split_mode must be 'median' or 'quartile'")

n_high = (df["group"] == "High").sum()
n_low = (df["group"] == "Low").sum()
if n_high < min_group_n or n_low < min_group_n:
    raise ValueError(f"Too few patients (High={n_high}, Low={n_low})")

# KM fit
kmf_high = KaplanMeierFitter(alpha=km_alpha)
kmf_low = KaplanMeierFitter(alpha=km_alpha)
h = df[df["group"] == "High"]
l = df[df["group"] == "Low"]
kmf_high.fit(h["_time"], h["_event"], label="High")
kmf_low.fit(l["_time"], l["_event"], label="Low")
lr = logrank_test(h["_time"], l["_time"], h["_event"], l["_event"])

# RMST via survRM2 (adjusted + unadjusted)
df["arm"] = (df["group"] == "High").astype(int)
r_df = df[["_time", "_event", "arm"] + cov_names].copy()
r_df.columns = ["time", "event", "arm"] + cov_names

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
ro.globalenv["tau"] = tau_years

ro.r("""
rmst_unadj <- rmst2(time=surv_data$time, status=surv_data$event,
                     arm=surv_data$arm, tau=tau)
""")

adjust = len(cov_names) > 0
if adjust:
    cov_r_str = ", ".join([f'"{c}"' for c in cov_names])
    ro.r(f"""
    cov_mat <- as.matrix(surv_data[, c({cov_r_str})])
    rmst_adj <- rmst2(time=surv_data$time, status=surv_data$event,
                      arm=surv_data$arm, tau=tau, covariates=cov_mat)
    """)
    adj = ro.r("rmst_adj$adjusted.result[1,]")
    adj_diff, adj_lo, adj_hi, adj_p = adj[0], adj[1], adj[2], adj[3]

unadj = ro.r("rmst_unadj$unadjusted.result[1,]")
unadj_diff, unadj_lo, unadj_hi, unadj_p = unadj[0], unadj[1], unadj[2], unadj[3]
rmst_h = ro.r("rmst_unadj$RMST.arm1$rmst['Est.']")[0]
rmst_l = ro.r("rmst_unadj$RMST.arm0$rmst['Est.']")[0]

# === FIGURE ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# Panel A: KM curves
for kmf, color, grp, n in [(kmf_high, COL_HIGH, "High", n_high), (kmf_low, COL_LOW, "Low", n_low)]:
    surv = kmf.survival_function_
    ci = kmf.confidence_interval_survival_function_
    t = np.concatenate([[0], surv.index.values])
    s = np.concatenate([[1], surv.iloc[:, 0].values])
    lo = np.concatenate([[1], ci.iloc[:, 0].values])
    hi = np.concatenate([[1], ci.iloc[:, 1].values])
    ax1.step(t, s, where="post", color=color, linewidth=2, label=f"{grp} (n={n})")
    ax1.fill_between(t, lo, hi, step="post", alpha=0.15, color=color)

ax1.grid(False)
ax1.set_xlabel(time_label, fontsize=11)
ax1.set_ylabel("Survival probability", fontsize=11)
ax1.set_ylim(0, 1.05)
ax1.set_xlim(0, None)
ax1.legend(frameon=False, fontsize=10, loc="lower left")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.set_title(f"Kaplan-Meier: Neighborhood {nh_col}\nlog-rank p = {lr.p_value:.3g}, {split_note}",
              fontsize=11, fontweight="bold")

# Panel B: RMST shaded area
ro.r("km_fit <- survfit(Surv(time, event) ~ arm, data = surv_data)")
km_t0 = np.concatenate([[0], np.array(ro.r("km_fit[1]$time"))])
km_s0 = np.concatenate([[1], np.array(ro.r("km_fit[1]$surv"))])
km_t1 = np.concatenate([[0], np.array(ro.r("km_fit[2]$time"))])
km_s1 = np.concatenate([[1], np.array(ro.r("km_fit[2]$surv"))])

# Truncate at tau
def truncate(t, s, tau):
    mask = t <= tau
    t_tr, s_tr = t[mask], s[mask]
    if t_tr[-1] < tau:
        t_tr = np.append(t_tr, tau)
        s_tr = np.append(s_tr, s_tr[-1])
    return t_tr, s_tr

t1, s1 = truncate(km_t1, km_s1, tau_years)
t0, s0 = truncate(km_t0, km_s0, tau_years)

ax2.grid(False)
ax2.step(t1, s1, where="post", color=COL_HIGH, linewidth=2, label=f"High (n={n_high})")
ax2.step(t0, s0, where="post", color=COL_LOW, linewidth=2, label=f"Low (n={n_low})")
ax2.fill_between(t1, s1, step="post", alpha=0.15, color=COL_HIGH)
ax2.fill_between(t0, s0, step="post", alpha=0.15, color=COL_LOW)
ax2.axvline(tau_years, ls="--", color="gray", alpha=0.6, lw=1)

# Stats text
cov_label = " + ".join(cov_names)
if adjust:
    txt = (f"RMST (τ = {tau_years} yr)\n"
           f"High: {rmst_h:.2f} yr\n"
           f"Low:  {rmst_l:.2f} yr\n"
           f"Δ = {adj_diff*12:+.1f} months\n"
           f"95% CI [{adj_lo*12:.1f}, {adj_hi*12:.1f}] mo\n"
           f"p (adj: {cov_label}) = {adj_p:.4f}\n"
           f"p (unadj) = {unadj_p:.4f}")
else:
    txt = (f"RMST (τ = {tau_years} yr)\n"
           f"High: {rmst_h:.2f} yr\n"
           f"Low:  {rmst_l:.2f} yr\n"
           f"Δ = {unadj_diff*12:+.1f} months\n"
           f"95% CI [{unadj_lo*12:.1f}, {unadj_hi*12:.1f}] mo\n"
           f"p = {unadj_p:.4f}")

ax2.text(0.97, 0.97, txt, transform=ax2.transAxes, fontsize=9,
         va="top", ha="right",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, lw=0.5, edgecolor="0.8"))

ax2.set_xlabel(time_label, fontsize=11)
ax2.set_ylabel("Survival probability", fontsize=11)
ax2.set_ylim(0, 1.05)
ax2.set_xlim(0, max(df["_time"].max(), tau_years) * 1.02)
ax2.legend(frameon=False, fontsize=10, loc="lower left")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_title(f"RMST: Neighborhood {nh_col}\n{cov_label} adjusted" if adjust else f"RMST: Neighborhood {nh_col}",
              fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(f"ECM_nhood_{nh_col}_{endpoint}_KM_RMST.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary
print(f"\n{endpoint} - ECM Neighborhood {nh_col}")
print(f"N = {len(df)} (High={n_high}, Low={n_low}), events={df['_event'].sum()}")
print(f"Log-rank p = {lr.p_value:.4g}")
print(f"RMST High = {rmst_h:.3f} yr, Low = {rmst_l:.3f} yr")
if adjust:
    print(f"RMST Δ (adj) = {adj_diff*12:+.1f} months, 95% CI [{adj_lo*12:.1f}, {adj_hi*12:.1f}], p = {adj_p:.4f}")
print(f"RMST Δ (unadj) = {unadj_diff*12:+.1f} months, 95% CI [{unadj_lo*12:.1f}, {unadj_hi*12:.1f}], p = {unadj_p:.4f}")

### Save ECM neighborhood data

In [ ]:
import os
import pandas as pd

# path to save
out_dir = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "ecm_neighborhood_annotations_v4.csv")

# UMAP for ECM neighborhoods
umap_ecm = adata.obsm["X_umap_ECM"]

# build table, indexed by spot id
df_out = pd.DataFrame(
    {
        "sample": adata.obs["sample"].astype(str),
        "ECM_leiden": adata.obs["ECM_leiden"].astype(str),
        "ECM_neighborhood": adata.obs["ECM_neighborhood"].astype(str),
        "UMAP1_ECM": umap_ecm[:, 0],
        "UMAP2_ECM": umap_ecm[:, 1],
    },
    index=adata.obs_names,
)
df_out.index.name = "spot_id"

df_out.to_csv(out_path)
print(f"Saved ECM annotations for {df_out.shape[0]} spots to:\n{out_path}")

### Reload ECM data

In [ ]:
import pandas as pd

in_path = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

df_ecm = pd.read_csv(in_path, index_col="spot_id")

# align to current object; will error if names disagree, which is what you want
df_ecm = df_ecm.loc[adata.obs_names]

# add back to obs and obsm
adata.obs["ECM_leiden"] = df_ecm["ECM_leiden"].astype("category")
adata.obs["ECM_neighborhood"] = df_ecm["ECM_neighborhood"].astype("category")
adata.obsm["X_umap_ECM"] = df_ecm[["UMAP1_ECM", "UMAP2_ECM"]].to_numpy()

print("Merged ECM_leiden, ECM_neighborhood, and X_umap_ECM into adata.")

### ECM neighborhood gene signatures 

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

nhood_key = "ECM_neighborhood"
LAYER = "log_norm_count"
MIN_SPOTS_PER_PSEUDOBULK = 30   # min spots for a patient-neighborhood combo
MIN_PATIENTS_PER_NHOOD = 10     # neighborhood must appear in this many patients
TOP_N_FULL = 10
TOP_N_SHORT = 5
LOGFC_THRESH = 0.25
FDR_THRESH = 0.05

# Which gene sets to test
ECM_ONLY = True   # True = ECM panel only, False = full transcriptome

if ECM_ONLY:
    test_genes = [g for g in ecm_genes_brain if g in adata.var_names]
    label = "ECM-restricted"
else:
    test_genes = list(adata.var_names)
    label = "Full transcriptome"

print(f"Running DE on {len(test_genes)} genes ({label})")

# 1) Pseudobulk: mean expression per patient per neighborhood
samples = adata.obs["sample"].values
nhoods = adata.obs[nhood_key].astype(str).values

# Extract expression matrix for test genes
gene_idx = adata.var_names.get_indexer(test_genes)
X = adata.layers[LAYER][:, gene_idx]
if issparse(X):
    X = X.toarray()

# Build pseudobulk table
pb_records = []
for sample_id in np.unique(samples):
    for nh in np.unique(nhoods):
        mask = (samples == sample_id) & (nhoods == nh)
        n_spots = mask.sum()
        if n_spots < MIN_SPOTS_PER_PSEUDOBULK:
            continue
        mean_expr = X[mask].mean(axis=0)
        pb_records.append({
            "sample": sample_id,
            "neighborhood": nh,
            "n_spots": n_spots,
            "expr": mean_expr,
        })

pb_df = pd.DataFrame(pb_records)
print(f"Pseudobulk entries: {len(pb_df)}")
print(f"Patients per neighborhood:")
print(pb_df.groupby("neighborhood")["sample"].nunique().sort_values(ascending=False))

# Filter neighborhoods with enough patients
nh_counts = pb_df.groupby("neighborhood")["sample"].nunique()
valid_nhoods = nh_counts[nh_counts >= MIN_PATIENTS_PER_NHOOD].index.tolist()
pb_df = pb_df[pb_df["neighborhood"].isin(valid_nhoods)]
print(f"\nNeighborhoods passing patient filter: {len(valid_nhoods)}")

# 2) One-vs-rest Wilcoxon on pseudobulked values
expr_matrix = np.vstack(pb_df["expr"].values)  # (n_pseudobulks x n_genes)
neighborhoods = pb_df["neighborhood"].values

all_results = []

for nh in sorted(valid_nhoods):
    mask_in = neighborhoods == nh
    mask_out = ~mask_in
    
    n_in = mask_in.sum()
    n_out = mask_out.sum()
    
    if n_in < MIN_PATIENTS_PER_NHOOD:
        continue
    
    X_in = expr_matrix[mask_in]
    X_out = expr_matrix[mask_out]
    
    pvals = []
    logfcs = []
    
    for j in range(len(test_genes)):
        vals_in = X_in[:, j]
        vals_out = X_out[:, j]
        
        lfc = vals_in.mean() - vals_out.mean()
        
        try:
            stat, p = ranksums(vals_in, vals_out)
        except Exception:
            p = 1.0
        
        pvals.append(p)
        logfcs.append(lfc)
    
    pvals = np.array(pvals)
    logfcs = np.array(logfcs)
    
    # FDR correction
    reject, fdr, _, _ = multipletests(pvals, method="fdr_bh")
    
    de_df = pd.DataFrame({
        "gene": test_genes,
        "logFC": logfcs,
        "pval": pvals,
        "fdr": fdr,
        "mean_in": X_in.mean(axis=0),
        "mean_out": X_out.mean(axis=0),
    })
    
    # Filter and rank
    sig = de_df[(de_df["fdr"] < FDR_THRESH) & (de_df["logFC"] > LOGFC_THRESH)].copy()
    sig = sig.sort_values(["logFC"], ascending=False)
    
    top10 = sig.head(TOP_N_FULL)["gene"].tolist()
    top5 = sig.head(TOP_N_SHORT)["gene"].tolist()
    
    all_results.append({
        "neighborhood": nh,
        "n_pseudobulks": n_in,
        "n_sig_genes": len(sig),
        "top10": top10,
        "top5": top5,
        "de_full": de_df.sort_values("pval"),
    })
    
    print(f"\nNeighborhood {nh} (n={n_in} pseudobulks, {len(sig)} sig genes):")
    print(f"  Top 10: {', '.join(top10) if top10 else 'None pass filters'}")
    print(f"  Top 5:  {', '.join(top5) if top5 else 'None pass filters'}")

# 3) Summary table
print(f"\n{'='*90}")
print(f"ECM NEIGHBORHOOD SIGNATURES ({label})")
print(f"{'='*90}")
header = f"{'Nhood':<8} {'N_pb':>5} {'N_sig':>6} | {'Top 5 markers':<45} | {'Top 10 markers'}"
print(header)
print("-" * len(header))

for r in sorted(all_results, key=lambda x: x["neighborhood"]):
    t5 = ", ".join(r["top5"]) if r["top5"] else "---"
    t10 = ", ".join(r["top10"]) if r["top10"] else "---"
    print(f"{r['neighborhood']:<8} {r['n_pseudobulks']:>5} {r['n_sig_genes']:>6} | {t5:<45} | {t10}")

# 4) Check for shared genes across top-5 lists
from collections import Counter
all_top5 = [g for r in all_results for g in r["top5"]]
dupes = {g: c for g, c in Counter(all_top5).items() if c > 1}
print(f"\nShared genes across top-5 lists: {dupes if dupes else 'None'}")

# 5) Save full DE results
all_de = []
for r in all_results:
    df_de = r["de_full"].copy()
    df_de["neighborhood"] = r["neighborhood"]
    all_de.append(df_de)

pd.concat(all_de).to_csv(f"ECM_neighborhood_signatures_{label.replace(' ', '_')}.csv", index=False)
print(f"\nSaved full DE results")

### ECM gene signature using all genes

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

nhood_key = "ECM_neighborhood"
LAYER = "log_norm_count"
MIN_SPOTS_PER_PSEUDOBULK = 30
MIN_PATIENTS_PER_NHOOD = 10
TOP_N_FULL = 10
TOP_N_SHORT = 5
LOGFC_THRESH = 0.25
FDR_THRESH = 0.05

test_genes = list(adata.var_names)
print(f"Running DE on {len(test_genes)} genes (full transcriptome)")

samples = adata.obs["sample"].values
nhoods = adata.obs[nhood_key].astype(str).values

X = adata.layers[LAYER]

unique_samples = np.unique(samples)
unique_nhoods = np.unique(nhoods)

pb_list = []
pb_meta = []

for nh in unique_nhoods:
    for sid in unique_samples:
        mask = (samples == sid) & (nhoods == nh)
        n_spots = mask.sum()
        if n_spots < MIN_SPOTS_PER_PSEUDOBULK:
            continue
        
        X_sub = X[mask]
        if issparse(X_sub):
            mean_expr = np.asarray(X_sub.mean(axis=0)).ravel()
        else:
            mean_expr = X_sub.mean(axis=0)
        
        pb_list.append(mean_expr)
        pb_meta.append({"sample": sid, "neighborhood": nh, "n_spots": n_spots})

pb_matrix = np.vstack(pb_list)
pb_df = pd.DataFrame(pb_meta)

print(f"Pseudobulk matrix: {pb_matrix.shape}")
print(f"Patients per neighborhood:")
print(pb_df.groupby("neighborhood")["sample"].nunique().sort_values(ascending=False))

nh_counts = pb_df.groupby("neighborhood")["sample"].nunique()
valid_nhoods = nh_counts[nh_counts >= MIN_PATIENTS_PER_NHOOD].index.tolist()
print(f"Neighborhoods passing filter: {len(valid_nhoods)}")

neighborhoods = pb_df["neighborhood"].values

all_results_full = []

for nh in sorted(valid_nhoods):
    mask_in = neighborhoods == nh
    mask_out = ~mask_in
    
    n_in = mask_in.sum()
    if n_in < MIN_PATIENTS_PER_NHOOD:
        continue
    
    X_in = pb_matrix[mask_in]
    X_out = pb_matrix[mask_out]
    
    mean_in = X_in.mean(axis=0)
    mean_out = X_out.mean(axis=0)
    logfcs = mean_in - mean_out
    
    pvals = np.ones(len(test_genes))
    for j in range(len(test_genes)):
        try:
            _, pvals[j] = ranksums(X_in[:, j], X_out[:, j])
        except Exception:
            pass
    
    _, fdr, _, _ = multipletests(pvals, method="fdr_bh")
    
    de_df = pd.DataFrame({
        "gene": test_genes,
        "logFC": logfcs,
        "pval": pvals,
        "fdr": fdr,
        "mean_in": mean_in,
        "mean_out": mean_out,
    })
    
    sig_up = de_df[(de_df["fdr"] < FDR_THRESH) & (de_df["logFC"] > LOGFC_THRESH)].copy()
    sig_up = sig_up.sort_values("logFC", ascending=False)
    
    top10 = sig_up.head(TOP_N_FULL)["gene"].tolist()
    top5 = sig_up.head(TOP_N_SHORT)["gene"].tolist()
    
    sig_down = de_df[(de_df["fdr"] < FDR_THRESH) & (de_df["logFC"] < -LOGFC_THRESH)].copy()
    sig_down = sig_down.sort_values("logFC", ascending=True)
    
    top5_down = sig_down.head(TOP_N_SHORT)["gene"].tolist()
    
    all_results_full.append({
        "neighborhood": nh,
        "n_pseudobulks": n_in,
        "n_sig_up": len(sig_up),
        "n_sig_down": len(sig_down),
        "top10": top10,
        "top5": top5,
        "top5_down": top5_down,
        "de_full": de_df.sort_values("pval"),
    })
    
    print(f"\nNeighborhood {nh} (n={n_in}, {len(sig_up)} up, {len(sig_down)} down):")
    print(f"  Top 5 UP:   {', '.join(top5) if top5 else '---'}")
    print(f"  Top 5 DOWN: {', '.join(top5_down) if top5_down else '---'}")
    print(f"  Top 10 UP:  {', '.join(top10) if top10 else '---'}")

print(f"\n{'='*100}")
print(f"FULL TRANSCRIPTOME SIGNATURES")
print(f"{'='*100}")
for r in sorted(all_results_full, key=lambda x: x["neighborhood"]):
    t5 = ", ".join(r["top5"]) if r["top5"] else "---"
    t5d = ", ".join(r["top5_down"]) if r["top5_down"] else "---"
    print(f"\nNhood {r['neighborhood']} ({r['n_pseudobulks']} pb, {r['n_sig_up']} up, {r['n_sig_down']} down):")
    print(f"  UP:   {t5}")
    print(f"  DOWN: {t5d}")

from collections import Counter
all_top5 = [g for r in all_results_full for g in r["top5"]]
dupes = {g: c for g, c in Counter(all_top5).items() if c > 1}
print(f"\nShared genes in top-5 UP: {dupes if dupes else 'None'}")

all_de = []
for r in all_results_full:
    df_de = r["de_full"].copy()
    df_de["neighborhood"] = r["neighborhood"]
    all_de.append(df_de)

pd.concat(all_de).to_csv("ECM_neighborhood_signatures_full_transcriptome.csv", index=False)
print("Saved: ECM_neighborhood_signatures_full_transcriptome.csv")

### Heatmap for ECM panel

In [ ]:
# %% [markdown]
# ### ECM gene expression per neighborhood — heatmap with dendrograms

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.sparse import issparse

# User config
nhood_key       = "ECM_neighborhood"
LAYER           = "log_norm_count"
CMAP            = "RdBu_r"
VMAX_PERCENTILE = 98
TOP_N_GENES     = 40             # for the focused-genes figure
SELECTION       = "variable"     # "variable" → top N by std across niches (bidirectional)
                                 # "per_niche" → top N per niche by |z|, union
ecm_csv_path    = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

# Load ECM neighborhoods from CSV if not already in adata.obs
if nhood_key not in adata.obs.columns:
    ecm_annot = pd.read_csv(ecm_csv_path).set_index("spot_id")
    adata.obs[nhood_key] = (
        ecm_annot[nhood_key]
        .reindex(adata.obs_names)
        .astype("category")
    )
    n_matched = adata.obs[nhood_key].notna().sum()
    print(f"Loaded ECM neighborhoods from CSV: {n_matched:,} of {adata.n_obs:,} spots matched")

# Filter to genes present in adata
ecm_genes_heatmap = [g for g in ecm_genes_brain if g in adata.var_names]
print(f"ECM genes available: {len(ecm_genes_heatmap)}")

# Mean expression per neighborhood
nhoods        = adata.obs[nhood_key].astype(str).values
valid         = ~pd.isna(adata.obs[nhood_key].values)
unique_nhoods = sorted([n for n in np.unique(nhoods[valid]) if n != "nan"],
                        key=lambda x: int(x))

gene_idx = adata.var_names.get_indexer(ecm_genes_heatmap)
X = adata.layers[LAYER][:, gene_idx]
if issparse(X):
    X = X.toarray()

nhood_means = {}
for nh in unique_nhoods:
    mask = (nhoods == nh) & valid
    nhood_means[nh] = X[mask].mean(axis=0)

mean_df = pd.DataFrame(nhood_means, index=ecm_genes_heatmap).T  # rows: niches, cols: genes
mean_z  = mean_df.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8), axis=0)


def draw_heatmap(genes_to_plot, fig_path, fig_width,
                 gene_fontsize=11, label_fontsize=11):
    """Clustered heatmap with aligned dendrograms for a subset of genes."""

    sub_z = mean_z[genes_to_plot]
    Z_r   = linkage(sub_z.values,    method="ward", metric="euclidean")
    Z_c   = linkage(sub_z.values.T,  method="ward", metric="euclidean")
    r_ord = leaves_list(Z_r)
    c_ord = leaves_list(Z_c)
    sub_z_ordered = sub_z.iloc[r_ord, c_ord]

    row_labels = [f"ECM_{sub_z.index[i]}" for i in r_ord]
    col_labels = [sub_z.columns[i]        for i in c_ord]

    vmax = np.percentile(np.abs(sub_z_ordered.values), VMAX_PERCENTILE)
    vmin = -vmax   # symmetric, diverging

    n_rows, n_cols = sub_z_ordered.shape
    fig_height = max(5.0, 0.35 * n_rows + 2.5)

    fig = plt.figure(figsize=(fig_width, fig_height))

    # Layout: [row labels | heatmap | row dendrogram]
    #         column dendrogram on top, aligned with heatmap column
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=[2.0, 30, 1.5],
        height_ratios=[1.2, 6],
        hspace=0.02, wspace=0.02,
        left=0.04, right=0.95, top=0.95, bottom=0.20,
    )

    # Top-left corner: empty (above row labels)
    ax_corner = fig.add_subplot(gs[0, 0]); ax_corner.axis("off")

    # Top: column dendrogram (above the heatmap column)
    ax_dendro_col = fig.add_subplot(gs[0, 1])
    dendrogram(Z_c, no_labels=True, ax=ax_dendro_col,
               color_threshold=0, above_threshold_color="0.4")
    # scipy draws leaves at 5, 15, 25, ... (5 + 10*i). To align leaf centers
    # with imshow cell centers (0, 1, 2, ...), the dendrogram x-range needs
    # to span exactly the same physical width as the heatmap, with leaves
    # centered in cells. xlim 0 to 10*n_cols achieves this when both axes
    # have identical width via the gridspec.
    ax_dendro_col.set_xlim(0, n_cols * 10)
    ax_dendro_col.axis("off")

    # Top-right corner: empty (above row dendrogram)
    ax_corner_r = fig.add_subplot(gs[0, 2]); ax_corner_r.axis("off")

    # Left: row labels
    ax_labels = fig.add_subplot(gs[1, 0])
    ax_labels.set_xlim(0, 1)
    ax_labels.set_ylim(n_rows - 0.5, -0.5)
    for i, lbl in enumerate(row_labels):
        ax_labels.text(0.95, i, lbl, ha="right", va="center",
                       fontsize=label_fontsize)
    ax_labels.axis("off")

    # Heatmap (center)
    ax = fig.add_subplot(gs[1, 1])
    im = ax.imshow(sub_z_ordered.values, aspect="auto", cmap=CMAP,
                   vmin=vmin, vmax=vmax, interpolation="none")
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_yticks([])
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels, rotation=45, fontsize=gene_fontsize,
                       ha="right", rotation_mode="anchor")
    ax.tick_params(axis="both", which="both", length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Right: row dendrogram (leaves point left, toward the heatmap)
    ax_dendro_row = fig.add_subplot(gs[1, 2])
    dendrogram(Z_r, orientation="right", no_labels=True, ax=ax_dendro_row,
               color_threshold=0, above_threshold_color="0.4")
    # scipy draws leaves at 5, 15, ...; match imshow's row centering and
    # invert so row 0 is at the top
    ax_dendro_row.set_ylim(0, n_rows * 10)
    ax_dendro_row.invert_yaxis()
    ax_dendro_row.axis("off")

    # Colorbar — placed past the row dendrogram on the right
    cbar = plt.colorbar(im, ax=ax_dendro_row, orientation="vertical",
                        shrink=0.6, pad=0.4, fraction=0.4)
    cbar.set_label("Z-score", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}  ({n_rows}×{n_cols})")
    return sub_z_ordered


# ---- Figure 1: full heatmap, all ECM genes ----
n_genes_full = len(ecm_genes_heatmap)
fig_w_full   = max(20.0, 0.32 * n_genes_full + 4.0)
print(f"\nFigure 1: full heatmap ({n_genes_full} genes, width={fig_w_full:.1f}\")")
draw_heatmap(
    genes_to_plot  = ecm_genes_heatmap,
    fig_path       = "ECM_neighborhood_heatmap_all.png",
    fig_width      = fig_w_full,
    gene_fontsize  = 12,
    label_fontsize = 13,
)

# ---- Figure 2: most-informative genes ----
if SELECTION == "variable":
    # Top N genes by standard deviation of z-score across niches.
    # These are the genes whose expression varies most between niches —
    # captures bidirectional signal (depletion AND enrichment).
    gene_var      = mean_z.std(axis=0)
    selected_genes = gene_var.sort_values(ascending=False).head(TOP_N_GENES).index.tolist()
    print(f"\nFigure 2: top {TOP_N_GENES} most-variable ECM genes (across niches)")
elif SELECTION == "per_niche":
    # Top N per niche by |z|, then union. Captures the strongest signed signal
    # per niche (enrichment or depletion) and combines.
    selected_genes = set()
    per_niche = max(3, TOP_N_GENES // len(unique_nhoods) + 2)
    for nh in unique_nhoods:
        top = mean_z.loc[nh].abs().sort_values(ascending=False).head(per_niche)
        selected_genes.update(top.index)
    selected_genes = list(selected_genes)
    print(f"\nFigure 2: top genes per niche (~{per_niche}/niche, "
          f"union → {len(selected_genes)} genes)")
else:
    raise ValueError(f"SELECTION must be 'variable' or 'per_niche', got {SELECTION!r}")

fig_w_top = max(12.0, 0.40 * len(selected_genes) + 4.0)
draw_heatmap(
    genes_to_plot  = selected_genes,
    fig_path       = f"ECM_neighborhood_heatmap_top{len(selected_genes)}.png",
    fig_width      = fig_w_top,
    gene_fontsize  = 18,
    label_fontsize = 18,
)

In [ ]:
# %% [markdown]
# ### ECM gene markers per neighborhood — composition-based heatmap
#
# Marker score computed on the COMPOSITION of each niche's ECM program,
# rather than on raw expression. This corrects for the fact that some
# niches (e.g. ECM_1, ECM_2) have globally lower ECM expression but
# nonetheless have a distinct compositional identity (collagen-dominated)
# that absolute-expression metrics miss.
#
# For each spot:
#   composition[niche, gene] = mean_expression[niche, gene]
#                              / sum(mean_expression[niche, :])
# So each niche's row sums to 1, regardless of total ECM activity.
# log2 FC and marker score are then computed in this compositional space.

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.sparse import issparse

# User config
nhood_key       = "ECM_neighborhood"
LAYER           = "log_norm_count"
CMAP            = "RdPu"
VMAX_PERCENTILE = 98
TOP_N_GENES     = 40
SELECTION       = "variable"     # "variable" → top N by marker-score range across niches
                                 # "per_niche" → top N per niche by marker score, union
ecm_csv_path    = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

# Load ECM neighborhoods if missing
if nhood_key not in adata.obs.columns:
    ecm_annot = pd.read_csv(ecm_csv_path).set_index("spot_id")
    adata.obs[nhood_key] = (
        ecm_annot[nhood_key]
        .reindex(adata.obs_names)
        .astype("category")
    )
    print(f"Loaded ECM neighborhoods: {adata.obs[nhood_key].notna().sum():,} spots")

ecm_genes_heatmap = [g for g in ecm_genes_brain if g in adata.var_names]
print(f"ECM genes available: {len(ecm_genes_heatmap)}")

# Mean expression per neighborhood
nhoods        = adata.obs[nhood_key].astype(str).values
valid         = ~pd.isna(adata.obs[nhood_key].values)
unique_nhoods = sorted([n for n in np.unique(nhoods[valid]) if n != "nan"],
                        key=lambda x: int(x))

gene_idx = adata.var_names.get_indexer(ecm_genes_heatmap)
X = adata.layers[LAYER][:, gene_idx]
if issparse(X):
    X = X.toarray()

nhood_means = {nh: X[(nhoods == nh) & valid].mean(axis=0) for nh in unique_nhoods}
mean_df     = pd.DataFrame(nhood_means, index=ecm_genes_heatmap).T   # rows: niches, cols: genes

# === COMPOSITIONAL NORMALIZATION ===
# Each row sums to 1 — fraction of niche's ECM expression contributed by each gene
row_sums    = mean_df.sum(axis=1).replace(0, np.nan)
composition = mean_df.div(row_sums, axis=0).fillna(0)

# Reference: average composition across niches
mean_comp = composition.mean(axis=0)

# log2 FC of composition: how much over- or under-represented is each gene
# in this niche's ECM program, vs the cohort-average ECM program
eps          = 1e-9
log2fc_comp  = np.log2((composition + eps).div(mean_comp + eps, axis=1))

# Marker score: composition × log2 FC⁺
# A gene that is a large fraction of this niche's ECM AND is over-represented
# relative to the cohort-average ECM gets a high score
marker_score = composition * log2fc_comp.clip(lower=0)

print(f"Compositional matrix: {composition.shape}")
print(f"Marker score range: [{marker_score.values.min():.4f}, {marker_score.values.max():.4f}]")


def draw_heatmap(genes_to_plot, fig_path, fig_width,
                 gene_fontsize=12, label_fontsize=13):
    """Clustered heatmap of compositional marker score with aligned dendrograms."""

    sub_M = marker_score[genes_to_plot]
    Z_r   = linkage(sub_M.values,    method="ward", metric="euclidean")
    Z_c   = linkage(sub_M.values.T,  method="ward", metric="euclidean")
    r_ord = leaves_list(Z_r)
    c_ord = leaves_list(Z_c)
    sub_M_ordered = sub_M.iloc[r_ord, c_ord]

    row_labels = [f"ECM_{sub_M.index[i]}" for i in r_ord]
    col_labels = [sub_M.columns[i]        for i in c_ord]

    vmax = np.percentile(sub_M_ordered.values, VMAX_PERCENTILE)
    vmin = 0

    n_rows, n_cols = sub_M_ordered.shape
    fig_height = max(5.0, 0.35 * n_rows + 2.5)

    fig = plt.figure(figsize=(fig_width, fig_height))

    # Layout: [row labels | heatmap | row dendrogram], col dendrogram on top
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=[2.0, 30, 1.5],
        height_ratios=[1.2, 6],
        hspace=0.02, wspace=0.02,
        left=0.04, right=0.95, top=0.95, bottom=0.20,
    )

    ax_corner = fig.add_subplot(gs[0, 0]); ax_corner.axis("off")

    # Top: column dendrogram
    ax_dendro_col = fig.add_subplot(gs[0, 1])
    dendrogram(Z_c, no_labels=True, ax=ax_dendro_col,
               color_threshold=0, above_threshold_color="0.4")
    ax_dendro_col.set_xlim(0, n_cols * 10)
    ax_dendro_col.axis("off")

    ax_corner_r = fig.add_subplot(gs[0, 2]); ax_corner_r.axis("off")

    # Left: row labels
    ax_labels = fig.add_subplot(gs[1, 0])
    ax_labels.set_xlim(0, 1)
    ax_labels.set_ylim(n_rows - 0.5, -0.5)
    for i, lbl in enumerate(row_labels):
        ax_labels.text(0.95, i, lbl, ha="right", va="center",
                       fontsize=label_fontsize)
    ax_labels.axis("off")

    # Center: heatmap
    ax = fig.add_subplot(gs[1, 1])
    im = ax.imshow(sub_M_ordered.values, aspect="auto", cmap=CMAP,
                   vmin=vmin, vmax=vmax, interpolation="none")
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_yticks([])
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels, rotation=45, fontsize=gene_fontsize,
                       ha="right", rotation_mode="anchor")
    ax.tick_params(axis="both", which="both", length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Right: row dendrogram (leaves point left, toward heatmap)
    ax_dendro_row = fig.add_subplot(gs[1, 2])
    dendrogram(Z_r, orientation="right", no_labels=True, ax=ax_dendro_row,
               color_threshold=0, above_threshold_color="0.4")
    ax_dendro_row.set_ylim(0, n_rows * 10)
    ax_dendro_row.invert_yaxis()
    ax_dendro_row.axis("off")

    # Colorbar past the row dendrogram
    cbar = plt.colorbar(im, ax=ax_dendro_row, orientation="vertical",
                        shrink=0.6, pad=0.4, fraction=0.4)
    cbar.set_label("Composition marker score\n(comp × log₂ FC⁺ of comp)",
                   fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}  ({n_rows}×{n_cols})")
    return sub_M_ordered


# ---- Figure 1: full heatmap ----
n_genes_full = len(ecm_genes_heatmap)
fig_w_full   = max(20.0, 0.32 * n_genes_full + 4.0)
print(f"\nFigure 1: full composition-marker heatmap ({n_genes_full} genes)")
draw_heatmap(
    genes_to_plot  = ecm_genes_heatmap,
    fig_path       = "ECM_neighborhood_compmarker_heatmap_all.png",
    fig_width      = fig_w_full,
    gene_fontsize  = 12,
    label_fontsize = 13,
)

# ---- Figure 2: most-informative genes ----
if SELECTION == "variable":
    gene_spread    = marker_score.max(axis=0) - marker_score.min(axis=0)
    selected_genes = gene_spread.sort_values(ascending=False).head(TOP_N_GENES).index.tolist()
    print(f"\nFigure 2: top {TOP_N_GENES} genes by composition-marker spread")
elif SELECTION == "per_niche":
    selected_genes = set()
    per_niche = max(3, TOP_N_GENES // len(unique_nhoods) + 2)
    for nh in unique_nhoods:
        top = marker_score.loc[nh].sort_values(ascending=False).head(per_niche)
        selected_genes.update(top.index)
    selected_genes = list(selected_genes)
    print(f"\nFigure 2: top {per_niche} per niche → {len(selected_genes)} genes")
else:
    raise ValueError(f"SELECTION must be 'variable' or 'per_niche', got {SELECTION!r}")

fig_w_top = max(12.0, 0.40 * len(selected_genes) + 4.0)
draw_heatmap(
    genes_to_plot  = selected_genes,
    fig_path       = f"ECM_neighborhood_compmarker_heatmap_top{len(selected_genes)}.png",
    fig_width      = fig_w_top,
    gene_fontsize  = 13,
    label_fontsize = 13,
)

adata.uns[f"{nhood_key}_composition"]      = composition
adata.uns[f"{nhood_key}_comp_log2fc"]      = log2fc_comp
adata.uns[f"{nhood_key}_comp_marker_score"] = marker_score
print(f"\nStored composition tables in adata.uns:")
print(f"  {nhood_key}_composition         (rows sum to 1 per niche)")
print(f"  {nhood_key}_comp_log2fc         (log2FC of composition)")
print(f"  {nhood_key}_comp_marker_score   (composition × log2FC⁺)")

### Test of ECM Association

In [ ]:
# %% [markdown]
# ### Diagnostic: TIL zone × ECM neighborhood × TIL group
#
# Investigates whether the bubble plot (patient-level mean differences)
# and the heatmap (spot-pooled fractions) tell different stories,
# specifically for ECM_8 in TIL+ zones.
#
# Three panels:
#   A. Per-patient ECM fractions, boxplots by TIL group within each zone
#   B. Spot-pooled vs patient-mean fractions, scatter
#   C. Patient counts per (zone × TIL group)

# %%
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# User config
EFS_ROOT      = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd")
TIL_CSV       = EFS_ROOT / "results" / "visium" / "til_zones" / "til_zone_annotations.csv"
ECM_CSV       = EFS_ROOT / "Analysis_Repo" / "Notebooks" / "ecm_neighborhood_annotations_v4.csv"
ZONES         = ["TIL+", "BZ1", "BZ2", "TIL-"]
ZONE_COLORS   = {"TIL+": "#c0392b", "BZ1": "#e67e22", "BZ2": "#3498db", "TIL-": "#2c3e50"}
ECM_NHOODS    = [str(i) for i in range(10)]
MIN_SPOTS     = 20         # min spots per (sample, zone) to include the patient
HIGHLIGHT_NHD = "8"        # ECM neighborhood to scrutinize visually
HIGHLIGHT_ZONE = "TIL+"

# ---- Load TIL zone annotations ----
til_df = pd.read_csv(TIL_CSV)
print(f"TIL zone CSV columns: {til_df.columns.tolist()}")
print(f"TIL zone CSV shape: {til_df.shape}")

# Try common column names
if "spot_id" in til_df.columns:
    til_df = til_df.set_index("spot_id")
elif til_df.columns[0] in ("Unnamed: 0", ""):
    til_df = til_df.set_index(til_df.columns[0])
print(f"TIL zone index sample: {til_df.index[:3].tolist()}")

# ---- Load ECM neighborhoods ----
ecm_df = pd.read_csv(ECM_CSV).set_index("spot_id")
print(f"\nECM CSV columns: {ecm_df.columns.tolist()}")

# ---- Build merged spot-level table ----
# Use whichever obs columns exist
obs = adata.obs[["sample"]].copy()

# TIL zone
if "TIL_zone" in adata.obs.columns:
    obs["TIL_zone"] = adata.obs["TIL_zone"].astype(str)
    print("\nUsing adata.obs['TIL_zone']")
else:
    zone_col = next((c for c in til_df.columns if "zone" in c.lower()), None)
    if zone_col is None:
        raise KeyError(f"No zone column found in TIL CSV: {til_df.columns.tolist()}")
    obs["TIL_zone"] = til_df[zone_col].reindex(obs.index).astype(str)
    print(f"Loaded TIL_zone from CSV column '{zone_col}'")

# TIL group (patient-level)
if "Sample_TIL_Group" in adata.obs.columns:
    obs["TIL_group"] = (
        adata.obs["Sample_TIL_Group"].astype(str)
        .str.replace("-", "_").str.replace("High", "high").str.replace("Low", "low")
    )
    print("Using adata.obs['Sample_TIL_Group']")
elif "TIL_group" in adata.obs.columns:
    obs["TIL_group"] = adata.obs["TIL_group"].astype(str)
    print("Using adata.obs['TIL_group']")
else:
    # Compute on the fly. TIL_positive is True for spots whose TIL_zone == "TIL+";
    # patient-level TIL_group is "TIL_high" if patient's TIL+ fraction is in
    # the top quartile of the cohort, else "TIL_low".
    print("Computing TIL_group on the fly from TIL_zone (top quartile by TIL+ fraction)")
    GROUP_KEY = "patient" if "patient" in adata.obs.columns else "sample"
    QUANTILE_CUTOFF = 0.75

    work_df = pd.DataFrame({
        GROUP_KEY: adata.obs[GROUP_KEY].values,
        "TIL_zone": obs["TIL_zone"].values,    # already loaded above
    })
    work_df["_til"] = (work_df["TIL_zone"] == "TIL+").astype(int)

    til_summary = work_df.groupby(GROUP_KEY, as_index=False).agg(
        TIL_spots=("_til", "sum"),
        Total_Spots=("_til", "size"),
    )
    til_summary["TIL_fraction"] = til_summary["TIL_spots"] / til_summary["Total_Spots"]
    cutoff = til_summary["TIL_fraction"].quantile(QUANTILE_CUTOFF)
    til_summary["TIL_group"] = np.where(
        til_summary["TIL_fraction"] >= cutoff, "TIL_high", "TIL_low"
    )

    group_map = til_summary.set_index(GROUP_KEY)["TIL_group"].to_dict()
    obs["TIL_group"] = adata.obs[GROUP_KEY].map(group_map)
    print(f"  Grouping key: {GROUP_KEY}")
    print(f"  Cutoff at Q={QUANTILE_CUTOFF}: TIL_fraction = {cutoff:.4f}")
    print(f"  TIL_high patients: {(til_summary['TIL_group'] == 'TIL_high').sum()}")
    print(f"  TIL_low patients:  {(til_summary['TIL_group'] == 'TIL_low').sum()}")

print(f"TIL_group values: {obs['TIL_group'].value_counts(dropna=False).to_dict()}")

# ECM neighborhood
obs["ECM_neighborhood"] = ecm_df["ECM_neighborhood"].reindex(obs.index).astype(str)

# Restrict to spots with all three annotations valid
valid = (
    obs["TIL_zone"].isin(ZONES)
    & obs["TIL_group"].isin(["TIL_high", "TIL_low"])
    & obs["ECM_neighborhood"].isin(ECM_NHOODS)
)
df = obs[valid].copy()
print(f"\nValid spots: {len(df):,} of {len(obs):,}")
print(f"Patients: {df['sample'].nunique()}")
print(df.groupby("TIL_group")["sample"].nunique().to_string())

# =====================================================================
# Per-sample per-zone ECM neighborhood fractions
# =====================================================================
counts = df.groupby(["sample", "TIL_zone", "ECM_neighborhood"], observed=True).size().reset_index(name="n_spots")
zone_totals = df.groupby(["sample", "TIL_zone"], observed=True).size().reset_index(name="zone_total")
counts = counts.merge(zone_totals, on=["sample", "TIL_zone"])
counts = counts[counts["zone_total"] >= MIN_SPOTS]
counts["fraction"] = counts["n_spots"] / counts["zone_total"]

# Add TIL group
sample_to_group = df.drop_duplicates("sample").set_index("sample")["TIL_group"]
counts["TIL_group"] = counts["sample"].map(sample_to_group)

# Fill missing (sample, zone, ecm) combinations with 0 for patients with that zone
# For each (sample, zone) that survived MIN_SPOTS, compute fraction for every ECM neighborhood
all_combos = []
for (sample, zone), grp in counts.groupby(["sample", "TIL_zone"]):
    zone_total = grp["zone_total"].iloc[0]
    til_group = grp["TIL_group"].iloc[0]
    nhd_counts = dict(zip(grp["ECM_neighborhood"], grp["n_spots"]))
    for nhd in ECM_NHOODS:
        all_combos.append({
            "sample": sample, "TIL_zone": zone, "ECM_neighborhood": nhd,
            "n_spots": nhd_counts.get(nhd, 0), "zone_total": zone_total,
            "fraction": nhd_counts.get(nhd, 0) / zone_total, "TIL_group": til_group,
        })
patient_fracs = pd.DataFrame(all_combos)

# =====================================================================
# Panel C: Patient counts per (zone × TIL group)
# =====================================================================
patient_counts = (
    patient_fracs.drop_duplicates(["sample", "TIL_zone"])
    .groupby(["TIL_zone", "TIL_group"], observed=True).size().unstack(fill_value=0)
    .reindex(ZONES)
)
print(f"\nPatient counts per (zone × TIL group), min {MIN_SPOTS} spots:")
print(patient_counts.to_string())

# =====================================================================
# Panel B: spot-pooled vs patient-mean fractions
# =====================================================================
# Spot-pooled: pool all spots in (zone, TIL_group), count ECM_X / total
spot_pool_records = []
patient_mean_records = []
for zone in ZONES:
    for grp in ["TIL_high", "TIL_low"]:
        sub = df[(df["TIL_zone"] == zone) & (df["TIL_group"] == grp)]
        total = len(sub)
        if total == 0:
            continue
        for nhd in ECM_NHOODS:
            spot_frac = (sub["ECM_neighborhood"] == nhd).sum() / total
            spot_pool_records.append({
                "TIL_zone": zone, "TIL_group": grp, "ECM_neighborhood": nhd,
                "spot_pool_frac": spot_frac,
            })
        # Patient mean
        sub_pf = patient_fracs[
            (patient_fracs["TIL_zone"] == zone) & (patient_fracs["TIL_group"] == grp)
        ]
        for nhd in ECM_NHOODS:
            patient_mean = sub_pf[sub_pf["ECM_neighborhood"] == nhd]["fraction"].mean()
            patient_mean_records.append({
                "TIL_zone": zone, "TIL_group": grp, "ECM_neighborhood": nhd,
                "patient_mean_frac": patient_mean,
            })
spot_pool_df    = pd.DataFrame(spot_pool_records)
patient_mean_df = pd.DataFrame(patient_mean_records)
compare_df = spot_pool_df.merge(patient_mean_df, on=["TIL_zone", "TIL_group", "ECM_neighborhood"])
compare_df["diff"] = compare_df["spot_pool_frac"] - compare_df["patient_mean_frac"]

# =====================================================================
# Figure with three panels
# =====================================================================
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 1, height_ratios=[1.6, 1.0, 0.8], hspace=0.35)

# ---- Panel A: per-patient fractions, box+strip per zone × ecm × group ----
ax_a = fig.add_subplot(gs[0])

n_zones = len(ZONES)
n_nhds  = len(ECM_NHOODS)
total_w = 0.85
group_w = total_w / 2
positions = []
for zi, zone in enumerate(ZONES):
    for ni, nhd in enumerate(ECM_NHOODS):
        x_center = zi * (n_nhds + 1.5) + ni
        positions.append((zone, nhd, x_center))

box_data_high = []
box_data_low  = []
box_pos_high  = []
box_pos_low   = []
strip_data    = []  # (x, y, group)

for zone, nhd, x in positions:
    for grp, off, store_data, store_pos in [
        ("TIL_high", -group_w/2, box_data_high, box_pos_high),
        ("TIL_low",   group_w/2, box_data_low,  box_pos_low),
    ]:
        sub = patient_fracs[
            (patient_fracs["TIL_zone"] == zone)
            & (patient_fracs["ECM_neighborhood"] == nhd)
            & (patient_fracs["TIL_group"] == grp)
        ]
        vals = sub["fraction"].values
        if len(vals) > 0:
            store_data.append(vals)
            store_pos.append(x + off)
            jitter = (np.random.RandomState(42).rand(len(vals)) - 0.5) * (group_w * 0.6)
            for v, j in zip(vals, jitter):
                strip_data.append((x + off + j, v, grp))

# Boxes
bp_high = ax_a.boxplot(box_data_high, positions=box_pos_high, widths=group_w * 0.85,
                       patch_artist=True, showfliers=False, zorder=2)
bp_low  = ax_a.boxplot(box_data_low,  positions=box_pos_low,  widths=group_w * 0.85,
                       patch_artist=True, showfliers=False, zorder=2)
for box in bp_high["boxes"]:
    box.set(facecolor="#fadbd8", edgecolor="#c0392b", linewidth=0.8)
for box in bp_low["boxes"]:
    box.set(facecolor="#d6eaf8", edgecolor="#2874a6", linewidth=0.8)
for bp in [bp_high, bp_low]:
    for whisker in bp["whiskers"]:
        whisker.set(color="#555555", linewidth=0.6)
    for cap in bp["caps"]:
        cap.set(color="#555555", linewidth=0.6)
    for median in bp["medians"]:
        median.set(color="black", linewidth=1.2)

# Strip dots, with highlight on ECM_8 in TIL+
for x, y, grp in strip_data:
    color = "#c0392b" if grp == "TIL_high" else "#2874a6"
    ax_a.scatter(x, y, s=6, color=color, alpha=0.5, zorder=3, edgecolors="none")

# Highlight zone for the focus comparison
for zone, nhd, x in positions:
    if zone == HIGHLIGHT_ZONE and nhd == HIGHLIGHT_NHD:
        ax_a.axvspan(x - 0.45, x + 0.45, color="#fffacd", alpha=0.6, zorder=0)

# X axis: ECM neighborhood labels under each zone block
ax_a.set_xticks([x for _, _, x in positions])
ax_a.set_xticklabels(
    [f"E{nhd}" for _, nhd, _ in positions],
    fontsize=8, rotation=0,
)
# Top labels: zone group names, centered
for zi, zone in enumerate(ZONES):
    x_left  = zi * (n_nhds + 1.5)
    x_right = x_left + n_nhds - 1
    x_mid   = (x_left + x_right) / 2
    ax_a.text(x_mid, 1.02, zone, transform=ax_a.get_xaxis_transform(),
              ha="center", va="bottom", fontsize=12, fontweight="semibold",
              color=ZONE_COLORS.get(zone, "#000000"))
    # Vertical separators between zones
    if zi < len(ZONES) - 1:
        ax_a.axvline((x_right + (zi + 1) * (n_nhds + 1.5)) / 2,
                     color="#cccccc", linewidth=0.5, zorder=1)

ax_a.set_ylabel("Per-patient fraction", fontsize=11)
ax_a.set_xlabel("ECM neighborhood (within each TIL zone)", fontsize=11, labelpad=8)
ax_a.set_title(
    f"Panel A: Per-patient ECM neighborhood fractions by zone and TIL group "
    f"(highlighted: ECM_{HIGHLIGHT_NHD} in {HIGHLIGHT_ZONE})",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)
ax_a.grid(False)
for s in ["top", "right"]:
    ax_a.spines[s].set_visible(False)
ax_a.set_xlim(-1, max([x for _, _, x in positions]) + 1)
ax_a.set_ylim(0, max(0.05, patient_fracs["fraction"].quantile(0.99) * 1.1))

# Legend
legend_handles = [
    Patch(facecolor="#fadbd8", edgecolor="#c0392b", label="TIL-high patients"),
    Patch(facecolor="#d6eaf8", edgecolor="#2874a6", label="TIL-low patients"),
]
ax_a.legend(handles=legend_handles, loc="upper right", frameon=False, fontsize=10)

# ---- Panel B: spot-pool vs patient-mean ----
ax_b = fig.add_subplot(gs[1])
for grp, color in [("TIL_high", "#c0392b"), ("TIL_low", "#2874a6")]:
    sub = compare_df[compare_df["TIL_group"] == grp]
    ax_b.scatter(sub["patient_mean_frac"], sub["spot_pool_frac"],
                 s=40, color=color, alpha=0.6, edgecolors="#333333",
                 linewidths=0.4, label=grp.replace("_", "-"))
    # Highlight ECM_HIGHLIGHT_NHD in HIGHLIGHT_ZONE
    hl = sub[(sub["ECM_neighborhood"] == HIGHLIGHT_NHD) & (sub["TIL_zone"] == HIGHLIGHT_ZONE)]
    if len(hl) > 0:
        for _, row in hl.iterrows():
            ax_b.scatter(row["patient_mean_frac"], row["spot_pool_frac"],
                         s=200, facecolor="none", edgecolor="#f1c40f",
                         linewidths=2.5, zorder=10)
            ax_b.annotate(
                f"ECM_{row['ECM_neighborhood']} in {row['TIL_zone']} ({grp})",
                (row["patient_mean_frac"], row["spot_pool_frac"]),
                xytext=(8, 8), textcoords="offset points",
                fontsize=9, fontweight="semibold", color="#444444",
            )

mx = max(compare_df["patient_mean_frac"].max(), compare_df["spot_pool_frac"].max()) * 1.05
ax_b.plot([0, mx], [0, mx], color="#888888", linestyle="--", linewidth=1.0, label="y = x")
ax_b.set_xlim(0, mx)
ax_b.set_ylim(0, mx)
ax_b.set_xlabel("Patient-mean fraction (each patient counted once)", fontsize=11)
ax_b.set_ylabel("Spot-pooled fraction (each spot counted once)", fontsize=11)
ax_b.set_title(
    "Panel B: Spot-pooled vs patient-mean fractions — divergence reveals "
    "patient-weighting effects",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)
ax_b.legend(loc="upper left", frameon=False, fontsize=10)
ax_b.grid(False)
for s in ["top", "right"]:
    ax_b.spines[s].set_visible(False)

# ---- Panel C: Patient counts ----
ax_c = fig.add_subplot(gs[2])
x = np.arange(len(ZONES))
w = 0.38
high_counts = [patient_counts.loc[z, "TIL_high"] if "TIL_high" in patient_counts.columns else 0 for z in ZONES]
low_counts  = [patient_counts.loc[z, "TIL_low"]  if "TIL_low"  in patient_counts.columns else 0 for z in ZONES]
ax_c.bar(x - w/2, high_counts, w, color="#c0392b", label="TIL-high", edgecolor="white", linewidth=0.5)
ax_c.bar(x + w/2, low_counts,  w, color="#2874a6", label="TIL-low",  edgecolor="white", linewidth=0.5)
for i, (h, l) in enumerate(zip(high_counts, low_counts)):
    ax_c.text(i - w/2, h + 1, str(h), ha="center", va="bottom", fontsize=9)
    ax_c.text(i + w/2, l + 1, str(l), ha="center", va="bottom", fontsize=9)
ax_c.set_xticks(x)
ax_c.set_xticklabels(ZONES, fontsize=11)
ax_c.set_ylabel(f"Patients with ≥{MIN_SPOTS} spots", fontsize=11)
ax_c.set_title(
    f"Panel C: Patient counts contributing to each (zone × TIL group) "
    f"— asymmetry indicates outlier sensitivity",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)
ax_c.legend(loc="upper right", frameon=False, fontsize=10)
ax_c.grid(False)
for s in ["top", "right"]:
    ax_c.spines[s].set_visible(False)

plt.savefig("ecm_til_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()
print("\nSaved: ecm_til_diagnostic.png")

# =====================================================================
# Print explicit summary for ECM_HIGHLIGHT in HIGHLIGHT_ZONE
# =====================================================================
print(f"\n{'='*70}")
print(f"FOCUSED COMPARISON: ECM_{HIGHLIGHT_NHD} in {HIGHLIGHT_ZONE}")
print(f"{'='*70}")
focus = patient_fracs[
    (patient_fracs["TIL_zone"] == HIGHLIGHT_ZONE)
    & (patient_fracs["ECM_neighborhood"] == HIGHLIGHT_NHD)
]
for grp in ["TIL_high", "TIL_low"]:
    g = focus[focus["TIL_group"] == grp]
    print(f"\n  {grp}: n={len(g)} patients")
    print(f"    Patient-mean fraction:  {g['fraction'].mean():.4f}")
    print(f"    Patient-median fraction: {g['fraction'].median():.4f}")
    print(f"    Spot-pooled fraction:   "
          f"{compare_df[(compare_df['TIL_zone']==HIGHLIGHT_ZONE) & (compare_df['ECM_neighborhood']==HIGHLIGHT_NHD) & (compare_df['TIL_group']==grp)]['spot_pool_frac'].iloc[0]:.4f}")
    print(f"    Top 3 patients by fraction:")
    print(g.nlargest(3, "fraction")[["sample", "fraction", "n_spots", "zone_total"]].to_string(index=False))

# Save tables
patient_fracs.to_csv("patient_fracs_zone_ecm_group.csv", index=False)
compare_df.to_csv("spot_vs_patient_comparison.csv", index=False)
print("\nSaved tables: patient_fracs_zone_ecm_group.csv, spot_vs_patient_comparison.csv")

In [ ]:
# %% [markdown]
# ### Per-patient ECM neighborhood fractions, focus niches

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ZONES        = ["TIL+", "BZ1", "BZ2", "TIL-"]
FOCUS_NHDS   = ["0", "1", "2", "7", "8"]

# Colors matching bubble/heatmap RdBu_r palette extremes
COL_HIGH_FACE = "#f4c8c1"
COL_HIGH_EDGE = "#b2182b"
COL_LOW_FACE  = "#c5d9ec"
COL_LOW_EDGE  = "#2166ac"
COL_HIGH_DOT  = "#b2182b"
COL_LOW_DOT   = "#2166ac"

fig, ax = plt.subplots(figsize=(13, 5))

n_zones = len(ZONES)
n_nhds  = len(FOCUS_NHDS)
group_w = 0.42

positions = []
for zi, zone in enumerate(ZONES):
    for ni, nhd in enumerate(FOCUS_NHDS):
        x_center = zi * (n_nhds + 1.5) + ni
        positions.append((zone, nhd, x_center))

box_data_high, box_pos_high = [], []
box_data_low,  box_pos_low  = [], []
strip_data = []

for zone, nhd, x in positions:
    for grp, off, store_data, store_pos in [
        ("TIL_high", -group_w/2, box_data_high, box_pos_high),
        ("TIL_low",   group_w/2, box_data_low,  box_pos_low),
    ]:
        sub = patient_fracs[
            (patient_fracs["TIL_zone"] == zone)
            & (patient_fracs["ECM_neighborhood"] == nhd)
            & (patient_fracs["TIL_group"] == grp)
        ]
        vals = sub["fraction"].values
        if len(vals) > 0:
            store_data.append(vals)
            store_pos.append(x + off)
            jitter = (np.random.RandomState(42).rand(len(vals)) - 0.5) * (group_w * 0.5)
            for v, j in zip(vals, jitter):
                strip_data.append((x + off + j, v, grp))

bp_high = ax.boxplot(box_data_high, positions=box_pos_high, widths=group_w * 0.85,
                     patch_artist=True, showfliers=False, zorder=2)
bp_low  = ax.boxplot(box_data_low,  positions=box_pos_low,  widths=group_w * 0.85,
                     patch_artist=True, showfliers=False, zorder=2)
for box in bp_high["boxes"]:
    box.set(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, linewidth=0.9)
for box in bp_low["boxes"]:
    box.set(facecolor=COL_LOW_FACE, edgecolor=COL_LOW_EDGE, linewidth=0.9)
for bp in [bp_high, bp_low]:
    for w in bp["whiskers"]:
        w.set(color="#444444", linewidth=0.6)
    for c in bp["caps"]:
        c.set(color="#444444", linewidth=0.6)
    for m in bp["medians"]:
        m.set(color="black", linewidth=1.1)

for x, y, grp in strip_data:
    ax.scatter(x, y, s=4,
               color=COL_HIGH_DOT if grp == "TIL_high" else COL_LOW_DOT,
               alpha=0.45, zorder=3, edgecolors="none")

# X-axis: ECM_0 format, angled
ax.set_xticks([x for _, _, x in positions])
ax.set_xticklabels([f"ECM_{nhd}" for _, nhd, _ in positions],
                   fontsize=11, rotation=45, ha="right", rotation_mode="anchor")

# Zone labels above each block
for zi, zone in enumerate(ZONES):
    x_left  = zi * (n_nhds + 1.5)
    x_right = x_left + n_nhds - 1
    x_mid   = (x_left + x_right) / 2
    ax.text(x_mid, 1.04, zone, transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=12, fontweight="semibold",
            color="#222222")
    if zi < len(ZONES) - 1:
        ax.axvline((x_right + (zi + 1) * (n_nhds + 1.5)) / 2,
                   color="#dddddd", linewidth=0.5, zorder=1)

ax.set_ylabel("Per-patient ECM neighborhood fraction", fontsize=12)
ax.grid(False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", labelsize=10)
ax.set_xlim(-1, max([x for _, _, x in positions]) + 1)
ax.set_ylim(0, max(0.05,
    patient_fracs[patient_fracs["ECM_neighborhood"].isin(FOCUS_NHDS)]["fraction"].quantile(0.99) * 1.1
))

ax.legend(
    handles=[
        Patch(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
        Patch(facecolor=COL_LOW_FACE,  edgecolor=COL_LOW_EDGE,  label="TIL-low"),
    ],
    loc="upper right", frameon=False, fontsize=10,
)
plt.tight_layout()
plt.savefig("ecm_focus_niches_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: ecm_focus_niches_boxplot.png")

In [ ]:
# %% [markdown]
# ### Significance testing for focus-niches boxplot

# %%
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.multitest import multipletests

ZONES      = ["TIL+", "BZ1", "BZ2", "TIL-"]
FOCUS_NHDS = ["0", "1", "2", "7", "8"]

records = []
for zone in ZONES:
    for nhd in FOCUS_NHDS:
        h = patient_fracs[
            (patient_fracs["TIL_zone"] == zone)
            & (patient_fracs["ECM_neighborhood"] == nhd)
            & (patient_fracs["TIL_group"] == "TIL_high")
        ]["fraction"].dropna().values
        l = patient_fracs[
            (patient_fracs["TIL_zone"] == zone)
            & (patient_fracs["ECM_neighborhood"] == nhd)
            & (patient_fracs["TIL_group"] == "TIL_low")
        ]["fraction"].dropna().values

        if len(h) < 3 or len(l) < 3:
            records.append({
                "zone": zone, "ecm": f"ECM_{nhd}", "n_high": len(h), "n_low": len(l),
                "median_high": np.nan, "median_low": np.nan,
                "p_mw": np.nan, "p_welch": np.nan,
            })
            continue

        # Mann-Whitney U (preferred — robust to zero-inflation)
        try:
            _, p_mw = mannwhitneyu(h, l, alternative="two-sided")
        except ValueError:
            p_mw = np.nan

        # Welch's t (sensitivity check)
        try:
            _, p_welch = ttest_ind(h, l, equal_var=False)
        except ValueError:
            p_welch = np.nan

        records.append({
            "zone": zone, "ecm": f"ECM_{nhd}",
            "n_high": len(h), "n_low": len(l),
            "median_high": float(np.median(h)),
            "median_low":  float(np.median(l)),
            "p_mw":    p_mw,
            "p_welch": p_welch,
        })

df = pd.DataFrame(records)

# BH-FDR correction across all 20 tests, separately for each test
mask = df["p_mw"].notna()
if mask.any():
    _, fdr_mw, _, _ = multipletests(df.loc[mask, "p_mw"], method="fdr_bh")
    df.loc[mask, "fdr_mw"] = fdr_mw
mask = df["p_welch"].notna()
if mask.any():
    _, fdr_welch, _, _ = multipletests(df.loc[mask, "p_welch"], method="fdr_bh")
    df.loc[mask, "fdr_welch"] = fdr_welch

# Significance flag (BH-adjusted Mann-Whitney < 0.05)
def stars(q):
    if pd.isna(q):  return ""
    if q < 0.001:   return "***"
    if q < 0.01:    return "**"
    if q < 0.05:    return "*"
    return "ns"
df["sig"] = df["fdr_mw"].apply(stars)

# Direction
df["direction"] = np.where(
    df["median_high"] > df["median_low"], "high>low",
    np.where(df["median_high"] < df["median_low"], "low>high", "—")
)

# Print
print("="*100)
print("Per-niche TIL-high vs TIL-low: Mann-Whitney U with BH-FDR (5 niches × 4 zones = 20 tests)")
print("Welch's t-test included as sensitivity")
print("="*100)
fmt = df.copy()
for c in ["median_high", "median_low"]:
    fmt[c] = fmt[c].round(3)
for c in ["p_mw", "fdr_mw", "p_welch", "fdr_welch"]:
    fmt[c] = fmt[c].apply(lambda v: f"{v:.2e}" if pd.notna(v) else "—")
print(fmt[["zone", "ecm", "n_high", "n_low",
          "median_high", "median_low", "direction",
          "p_mw", "fdr_mw", "p_welch", "fdr_welch", "sig"]].to_string(index=False))

# Save
df.to_csv("focus_niches_significance.csv", index=False)
print("\nSaved: focus_niches_significance.csv")

# Quick concordance check
both_sig = ((df["fdr_mw"] < 0.05) & (df["fdr_welch"] < 0.05)).sum()
mw_only  = ((df["fdr_mw"] < 0.05) & ~(df["fdr_welch"] < 0.05)).sum()
welch_only = (~(df["fdr_mw"] < 0.05) & (df["fdr_welch"] < 0.05)).sum()
print(f"\nConcordance: both significant = {both_sig}, MW only = {mw_only}, Welch only = {welch_only}")

In [ ]:
# %% [markdown]
# ### Per-spot collagen expression: immune-engaged vs immune-cold zones
#
# Two figures:
#   1. Split by TIL group (TIL-high vs TIL-low patients shown separately)
#   2. Combined (one paired comparison across all patients)
#
# Within each patient, mean panel-collagen expression is computed in
# (a) immune-engaged spots: TIL+, BZ1, BZ2 combined
# (b) immune-cold spots: TIL- only
# Compared with paired Wilcoxon — each patient is their own control.

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.sparse import issparse
from scipy.stats import wilcoxon

# Config
COLLAGEN_PANEL  = ["COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
                   "COL5A1", "COL5A2", "COL6A1", "COL6A2", "COL6A3"]
LAYER           = "log_norm_count"
ENGAGED_ZONES   = ["TIL+", "BZ1", "BZ2"]
COLD_ZONES      = ["TIL-"]
MIN_SPOTS       = 20

# Pull collagen panel expression
def get_gene_matrix(genes):
    available = [g for g in genes if g in adata.var_names]
    missing   = set(genes) - set(available)
    if missing:
        print(f"Note — collagen genes not in adata: {sorted(missing)}")
    idx = adata.var_names.get_indexer(available)
    X = adata.layers[LAYER][:, idx]
    if issparse(X):
        X = X.toarray()
    return pd.DataFrame(X, index=adata.obs_names, columns=available)

gene_mat   = get_gene_matrix(COLLAGEN_PANEL)
panel_mean = gene_mat.mean(axis=1)
print(f"Panel: {gene_mat.shape[1]} collagens averaged per spot")

# Build working table — assumes 'obs' from earlier diagnostic in scope
work = pd.DataFrame({
    "sample":           obs["sample"].values,
    "TIL_zone":         obs["TIL_zone"].values,
    "TIL_group":        obs["TIL_group"].values,
    "collagen":         panel_mean.values,
}, index=adata.obs_names)
work["compartment"] = work["TIL_zone"].map(
    lambda z: "engaged" if z in ENGAGED_ZONES else ("cold" if z in COLD_ZONES else None)
)
work = work[work["compartment"].notna() & work["TIL_group"].isin(["TIL_high", "TIL_low"])]

# Per-patient mean per compartment
per_patient = (
    work.groupby(["sample", "TIL_group", "compartment"], observed=True)
        .agg(mean_collagen=("collagen", "mean"),
             n_spots=("collagen", "size"))
        .reset_index()
)
per_patient = per_patient[per_patient["n_spots"] >= MIN_SPOTS]

# Wide format — one row per patient with engaged + cold means
wide = per_patient.pivot_table(
    index=["sample", "TIL_group"], columns="compartment", values="mean_collagen"
).reset_index().dropna(subset=["engaged", "cold"])

# Print stats — split by group
print("\n" + "="*90)
print(f"Paired Wilcoxon: engaged ({'+'.join(ENGAGED_ZONES)}) vs cold ({COLD_ZONES[0]})")
print(f"Min {MIN_SPOTS} spots required per patient per compartment")
print("="*90)

stats_records = []
for grp_label, grp_filter in [
    ("TIL-high",         lambda d: d["TIL_group"] == "TIL_high"),
    ("TIL-low",          lambda d: d["TIL_group"] == "TIL_low"),
    ("Combined",         lambda d: d["TIL_group"].isin(["TIL_high", "TIL_low"])),
]:
    g = wide[grp_filter(wide)].copy()
    n = len(g)
    if n < 5:
        print(f"\n{grp_label}: too few patients ({n})")
        continue
    diff = (g["engaged"] - g["cold"]).values
    stat, p = wilcoxon(diff)
    pos = (diff > 0).sum()

    print(f"\n{grp_label} (n={n} paired patients):")
    print(f"  Engaged mean: {g['engaged'].mean():.3f}, median: {g['engaged'].median():.3f}")
    print(f"  Cold    mean: {g['cold'].mean():.3f}, median: {g['cold'].median():.3f}")
    print(f"  Per-patient (engaged − cold) mean: {diff.mean():.3f}, median: {np.median(diff):.3f}")
    print(f"  Wilcoxon paired p-value: {p:.2e}")
    print(f"  Patients with engaged > cold: {pos} of {n} ({pos/n*100:.0f}%)")

    stats_records.append({
        "group": grp_label, "n": n,
        "engaged_median": g["engaged"].median(),
        "cold_median":    g["cold"].median(),
        "delta_median":   np.median(diff),
        "p_wilcoxon":     p,
        "pct_directional": pos/n*100,
    })
stats_df = pd.DataFrame(stats_records)
stats_df.to_csv("collagen_engaged_vs_cold_stats.csv", index=False)

# =====================================================================
# Plot 1 — Split by TIL group
# =====================================================================
COL_HIGH_FACE = "#f4c8c1"
COL_HIGH_EDGE = "#b2182b"
COL_LOW_FACE  = "#c5d9ec"
COL_LOW_EDGE  = "#2166ac"

def paired_panel(ax, data, face_color, edge_color, title):
    """Draw paired engaged-vs-cold strip with connecting lines and box overlays."""
    n = len(data)
    if n == 0:
        ax.set_axis_off()
        return

    # Connecting lines
    for _, row in data.iterrows():
        ax.plot([0, 1], [row["cold"], row["engaged"]],
                color=edge_color, alpha=0.20, linewidth=0.6, zorder=2)

    # Strip dots
    rng = np.random.RandomState(42)
    j_cold    = rng.uniform(-0.06, 0.06, n)
    j_engaged = rng.uniform(-0.06, 0.06, n)
    ax.scatter(0 + j_cold,    data["cold"],    s=12, color=edge_color,
               alpha=0.55, zorder=3, edgecolors="none")
    ax.scatter(1 + j_engaged, data["engaged"], s=12, color=edge_color,
               alpha=0.55, zorder=3, edgecolors="none")

    # Boxes
    bp = ax.boxplot(
        [data["cold"].values, data["engaged"].values],
        positions=[0, 1], widths=0.45, patch_artist=True, showfliers=False, zorder=4,
    )
    for box in bp["boxes"]:
        box.set(facecolor=face_color, edgecolor=edge_color, linewidth=1.0, alpha=0.95)
    for w in bp["whiskers"]: w.set(color="#444444", linewidth=0.7)
    for c in bp["caps"]:     c.set(color="#444444", linewidth=0.7)
    for m in bp["medians"]:  m.set(color="black", linewidth=1.4)

    # Test
    diff = (data["engaged"] - data["cold"]).values
    _, p = wilcoxon(diff)
    pos = (diff > 0).sum()
    ax.text(0.5, 0.97,
            f"n = {n} patients\np = {p:.2e}\n{pos}/{n} engaged > cold",
            transform=ax.transAxes, ha="center", va="top",
            fontsize=9, color="#222222",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="#fafafa",
                      edgecolor="#cccccc", linewidth=0.5))

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["TIL−\n(immune-cold)",
                        "TIL+ / BZ1 / BZ2\n(immune-engaged)"], fontsize=10)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel("Per-patient mean collagen panel\n(log-normalized)", fontsize=11)
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.text(0.02, 0.98, title, transform=ax.transAxes, ha="left", va="top",
            fontsize=12, fontweight="semibold", color=edge_color)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
paired_panel(axes[0], wide[wide["TIL_group"] == "TIL_high"],
             COL_HIGH_FACE, COL_HIGH_EDGE, "TIL-high patients")
paired_panel(axes[1], wide[wide["TIL_group"] == "TIL_low"],
             COL_LOW_FACE,  COL_LOW_EDGE,  "TIL-low patients")
plt.tight_layout()
plt.savefig("collagen_engaged_vs_cold_split.png", dpi=300, bbox_inches="tight")
plt.show()
print("\nSaved: collagen_engaged_vs_cold_split.png")

# =====================================================================
# Plot 2 — Combined
# =====================================================================
COL_COMBO_FACE = "#dddddd"
COL_COMBO_EDGE = "#444444"

fig, ax = plt.subplots(figsize=(5, 5))
data_all = wide.copy()
n = len(data_all)

# Connecting lines colored by group
for _, row in data_all.iterrows():
    color = COL_HIGH_EDGE if row["TIL_group"] == "TIL_high" else COL_LOW_EDGE
    ax.plot([0, 1], [row["cold"], row["engaged"]],
            color=color, alpha=0.18, linewidth=0.6, zorder=2)

# Strip dots colored by group
rng = np.random.RandomState(42)
j_cold    = rng.uniform(-0.06, 0.06, n)
j_engaged = rng.uniform(-0.06, 0.06, n)
data_arr = data_all.reset_index(drop=True)
for i, row in data_arr.iterrows():
    color = COL_HIGH_EDGE if row["TIL_group"] == "TIL_high" else COL_LOW_EDGE
    ax.scatter(0 + j_cold[i],    row["cold"],    s=14, color=color,
               alpha=0.55, zorder=3, edgecolors="none")
    ax.scatter(1 + j_engaged[i], row["engaged"], s=14, color=color,
               alpha=0.55, zorder=3, edgecolors="none")

# Boxes (combined)
bp = ax.boxplot(
    [data_all["cold"].values, data_all["engaged"].values],
    positions=[0, 1], widths=0.45, patch_artist=True, showfliers=False, zorder=4,
)
for box in bp["boxes"]:
    box.set(facecolor=COL_COMBO_FACE, edgecolor=COL_COMBO_EDGE, linewidth=1.0, alpha=0.85)
for w in bp["whiskers"]: w.set(color="#444444", linewidth=0.7)
for c in bp["caps"]:     c.set(color="#444444", linewidth=0.7)
for m in bp["medians"]:  m.set(color="black", linewidth=1.4)

diff = (data_all["engaged"] - data_all["cold"]).values
_, p = wilcoxon(diff)
pos = (diff > 0).sum()
ax.text(0.5, 0.97,
        f"n = {n} patients\np = {p:.2e}\n{pos}/{n} engaged > cold",
        transform=ax.transAxes, ha="center", va="top",
        fontsize=9, color="#222222",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#fafafa",
                  edgecolor="#cccccc", linewidth=0.5))

ax.set_xticks([0, 1])
ax.set_xticklabels(["TIL−\n(immune-cold)",
                    "TIL+ / BZ1 / BZ2\n(immune-engaged)"], fontsize=10)
ax.set_xlim(-0.5, 1.5)
ax.set_ylabel("Per-patient mean collagen panel\n(log-normalized)", fontsize=11)
ax.grid(False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)

ax.legend(handles=[
    Patch(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
    Patch(facecolor=COL_LOW_FACE,  edgecolor=COL_LOW_EDGE,  label="TIL-low"),
], loc="lower right", frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig("collagen_engaged_vs_cold_combined.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: collagen_engaged_vs_cold_combined.png")

### ECM adjacency

In [ ]:
# Cell 2: Spatial adjacency / co-occurrence between ECM neighborhoods

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import squidpy as sq
from scipy.cluster.hierarchy import linkage, leaves_list
import gc

nhood_key = "ECM_neighborhood"
coord_type = "grid"
n_rings = 1

# Build contact frequency matrix across all samples
unique_nhoods = sorted(adata.obs[nhood_key].dropna().astype(str).unique())
n_nh = len(unique_nhoods)
nh_to_idx = {nh: i for i, nh in enumerate(unique_nhoods)}

contact_counts = np.zeros((n_nh, n_nh), dtype=np.float64)
samples = adata.obs["sample"].unique()

print(f"Computing spatial adjacency across {len(samples)} samples...")

for i, sid in enumerate(samples):
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(samples)}")
    
    ad_s = adata[adata.obs["sample"] == sid].copy()
    if ad_s.n_obs < 50:
        continue
    
    # Build spatial neighbors
    if "spatial" not in ad_s.obsm:
        if {"imagecol", "imagerow"}.issubset(ad_s.obs.columns):
            ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()
        else:
            continue
    
    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr()
    
    labels = ad_s.obs[nhood_key].astype(str).values
    
    # Count contacts
    rows, cols = A.nonzero()
    for r, c in zip(rows, cols):
        if r < c:  # each edge once
            nh_r = labels[r]
            nh_c = labels[c]
            if nh_r in nh_to_idx and nh_c in nh_to_idx:
                ir = nh_to_idx[nh_r]
                ic = nh_to_idx[nh_c]
                contact_counts[ir, ic] += 1
                contact_counts[ic, ir] += 1
    
    del ad_s, A
    gc.collect()

print("Done.")

# Normalize: observed / expected under random mixing
row_totals = contact_counts.sum(axis=1)
total = contact_counts.sum()
expected = np.outer(row_totals, row_totals) / (total + 1e-8)
enrichment = np.log2((contact_counts + 1) / (expected + 1))

# Make DataFrames
nh_labels = [f"N{nh}" for nh in unique_nhoods]
contact_df = pd.DataFrame(contact_counts, index=nh_labels, columns=nh_labels)
enrich_df = pd.DataFrame(enrichment, index=nh_labels, columns=nh_labels)

# Cluster order
Z = linkage(enrichment, method="ward", metric="euclidean")
order = leaves_list(Z)
ordered_labels = [nh_labels[i] for i in order]

enrich_ordered = enrich_df.loc[ordered_labels, ordered_labels]
contact_ordered = contact_df.loc[ordered_labels, ordered_labels]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Contact frequency (raw counts, log-scaled)
ax = axes[0]
ax.grid(False)
log_counts = np.log10(contact_ordered.values + 1)
im0 = ax.imshow(log_counts, cmap="viridis", aspect="equal", interpolation="none")
ax.set_xticks(range(len(ordered_labels)))
ax.set_xticklabels(ordered_labels, rotation=45, ha="right", fontsize=10, fontweight="bold")
ax.set_yticks(range(len(ordered_labels)))
ax.set_yticklabels(ordered_labels, fontsize=10, fontweight="bold")
ax.set_title("Spatial contact frequency\n(log10 counts)", fontsize=12, fontweight="bold")
cbar0 = plt.colorbar(im0, ax=ax, shrink=0.8)
cbar0.set_label("log10(contacts + 1)", fontsize=9)

# Panel B: Enrichment (log2 observed/expected)
ax = axes[1]
ax.grid(False)
vmax = np.percentile(np.abs(enrich_ordered.values), 95)
im1 = ax.imshow(enrich_ordered.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                aspect="equal", interpolation="none")
ax.set_xticks(range(len(ordered_labels)))
ax.set_xticklabels(ordered_labels, rotation=45, ha="right", fontsize=10, fontweight="bold")
ax.set_yticks(range(len(ordered_labels)))
ax.set_yticklabels(ordered_labels, fontsize=10, fontweight="bold")
ax.set_title("Spatial co-occurrence enrichment\n(log2 observed/expected)", fontsize=12, fontweight="bold")
cbar1 = plt.colorbar(im1, ax=ax, shrink=0.8)
cbar1.set_label("log2(O/E)", fontsize=9)

plt.tight_layout()
plt.savefig("ECM_neighborhood_spatial_adjacency.png", dpi=300, bbox_inches="tight")
plt.show()

# Print enrichment table
print("\nSpatial co-occurrence enrichment (log2 O/E):")
print(enrich_ordered.round(2).to_string())

# Top co-occurring pairs
pairs = []
for i in range(n_nh):
    for j in range(i+1, n_nh):
        pairs.append({
            "nhood_A": unique_nhoods[i],
            "nhood_B": unique_nhoods[j],
            "contacts": int(contact_counts[i, j]),
            "log2_enrichment": enrichment[i, j],
        })

pairs_df = pd.DataFrame(pairs).sort_values("log2_enrichment", ascending=False)
print("\nTop 10 co-occurring pairs:")
print(pairs_df.head(10).round(3).to_string(index=False))
print("\nTop 10 avoidant pairs:")
print(pairs_df.tail(10).round(3).to_string(index=False))

pairs_df.to_csv("ECM_neighborhood_spatial_adjacency.csv", index=False)
print("\nSaved: ECM_neighborhood_spatial_adjacency.csv")

### Nhood adjacency stats

In [ ]:
import numpy as np
import pandas as pd
import squidpy as sq
import gc

nhood_key = "ECM_neighborhood"
coord_type = "grid"
n_rings = 1
N_PERMS = 1000
SEED = 42

unique_nhoods = sorted(adata.obs[nhood_key].dropna().astype(str).unique())
n_nh = len(unique_nhoods)
nh_to_idx = {nh: i for i, nh in enumerate(unique_nhoods)}
samples = adata.obs["sample"].unique()

rng = np.random.default_rng(SEED)

# Precompute spatial graphs and labels per sample
print(f"Precomputing spatial graphs for {len(samples)} samples...")
sample_data = []

for i, sid in enumerate(samples):
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(samples)}")
    
    ad_s = adata[adata.obs["sample"] == sid].copy()
    if ad_s.n_obs < 50:
        continue
    
    if "spatial" not in ad_s.obsm:
        if {"imagecol", "imagerow"}.issubset(ad_s.obs.columns):
            ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()
        else:
            continue
    
    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr()
    labels = ad_s.obs[nhood_key].astype(str).values
    
    # Store edges (upper triangle only)
    rows, cols = A.nonzero()
    mask = rows < cols
    edges = np.column_stack([rows[mask], cols[mask]])
    
    sample_data.append({"labels": labels, "edges": edges})
    del ad_s, A
    gc.collect()

print(f"Stored graphs for {len(sample_data)} samples")


def compute_contacts(sample_data, permute=False, rng=None):
    counts = np.zeros((n_nh, n_nh), dtype=np.float64)
    for sd in sample_data:
        labels = sd["labels"].copy()
        if permute:
            rng.shuffle(labels)
        edges = sd["edges"]
        for r, c in edges:
            nh_r = labels[r]
            nh_c = labels[c]
            if nh_r in nh_to_idx and nh_c in nh_to_idx:
                ir = nh_to_idx[nh_r]
                ic = nh_to_idx[nh_c]
                counts[ir, ic] += 1
                counts[ic, ir] += 1
    return counts


# Observed
print("Computing observed contacts...")
obs_counts = compute_contacts(sample_data, permute=False)
obs_totals = obs_counts.sum(axis=1)
obs_total = obs_counts.sum()
obs_expected = np.outer(obs_totals, obs_totals) / (obs_total + 1e-8)
obs_enrichment = np.log2((obs_counts + 1) / (obs_expected + 1))

# Permutations
print(f"Running {N_PERMS} permutations...")
perm_enrichments = np.zeros((N_PERMS, n_nh, n_nh))

for p in range(N_PERMS):
    if (p + 1) % 100 == 0:
        print(f"  Permutation {p+1}/{N_PERMS}")
    
    perm_counts = compute_contacts(sample_data, permute=True, rng=rng)
    perm_totals = perm_counts.sum(axis=1)
    perm_total = perm_counts.sum()
    perm_expected = np.outer(perm_totals, perm_totals) / (perm_total + 1e-8)
    perm_enrichments[p] = np.log2((perm_counts + 1) / (perm_expected + 1))

# Two-sided p-values
p_values = np.zeros((n_nh, n_nh))
for i in range(n_nh):
    for j in range(n_nh):
        obs_val = obs_enrichment[i, j]
        perm_vals = perm_enrichments[:, i, j]
        if obs_val >= 0:
            p_values[i, j] = (np.sum(perm_vals >= obs_val) + 1) / (N_PERMS + 1)
        else:
            p_values[i, j] = (np.sum(perm_vals <= obs_val) + 1) / (N_PERMS + 1)

# DataFrames
nh_labels = [f"N{nh}" for nh in unique_nhoods]
enrich_df = pd.DataFrame(obs_enrichment, index=nh_labels, columns=nh_labels)
pval_df = pd.DataFrame(p_values, index=nh_labels, columns=nh_labels)

print("\nEnrichment (log2 O/E):")
print(enrich_df.round(2).to_string())
print("\nP-values:")
print(pval_df.round(4).to_string())

# Significant pairs
pairs = []
for i in range(n_nh):
    for j in range(i+1, n_nh):
        pairs.append({
            "nhood_A": unique_nhoods[i],
            "nhood_B": unique_nhoods[j],
            "log2_enrichment": obs_enrichment[i, j],
            "p_value": p_values[i, j],
            "contacts": int(obs_counts[i, j]),
        })

pairs_df = pd.DataFrame(pairs).sort_values("p_value")
pairs_df["significant"] = pairs_df["p_value"] < 0.05

print(f"\nSignificant co-occurring (enriched, p<0.05):")
sig_pos = pairs_df[(pairs_df["significant"]) & (pairs_df["log2_enrichment"] > 0)]
print(sig_pos.round(4).to_string(index=False))

print(f"\nSignificant avoidant (depleted, p<0.05):")
sig_neg = pairs_df[(pairs_df["significant"]) & (pairs_df["log2_enrichment"] < 0)]
print(sig_neg.round(4).to_string(index=False))

pairs_df.to_csv("ECM_neighborhood_spatial_adjacency_with_pvals.csv", index=False)
print("\nSaved: ECM_neighborhood_spatial_adjacency_with_pvals.csv")

### Summary output

In [ ]:
import numpy as np
import pandas as pd

nhood_counts = adata.obs["ECM_neighborhood"].value_counts().sort_index()
total_spots = nhood_counts.sum()

# Cell landscape means
if "means" not in dir():
    print("Recomputing cell type means...")
    c2l_prefix = "meanscell_abundance_w_sf_"
    curation_col = "celltype_level4_curated"
    celltype_key = adata.uns["celltype_annotation_key"]
    c2l_cols = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
    lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])
    counts_ct = adata.obs[c2l_cols].copy()
    counts_ct.columns = lvl4_names
    lvl4_in_key = lvl4_names.intersection(celltype_key.index)
    counts_ct = counts_ct[lvl4_in_key]
    curated_map_ct = celltype_key.loc[lvl4_in_key, curation_col].astype(str).fillna("Unknown")
    EXCLUDE_PATTERNS = ["doublet", "other"]
    keep_mask = ~curated_map_ct.str.lower().apply(lambda x: any(pat in x for pat in EXCLUDE_PATTERNS))
    counts_ct = counts_ct.loc[:, keep_mask.values]
    curated_map_ct = curated_map_ct[keep_mask]
    counts_curated = counts_ct.groupby(curated_map_ct, axis=1).sum()
    row_sums = counts_curated.sum(axis=1).replace(0, np.nan)
    frac_ct = counts_curated.div(row_sums, axis=0).fillna(0.0)
    nh_cat = adata.obs["ECM_neighborhood"].astype("category")
    means = frac_ct.groupby(nh_cat).mean()
    means = means.div(means.sum(axis=1), axis=0).fillna(0.0)

# Verify both signature results exist
_have_ecm = "all_results" in dir() and len(all_results) > 0
_have_full = "all_results_full" in dir() and len(all_results_full) > 0
_have_spearman = "df_res" in dir()
_have_rmst = "df_rmst" in dir()
_have_adjacency = "pairs_df" in dir()

print(f"Data availability: ECM sigs={_have_ecm}, Full sigs={_have_full}, "
      f"Spearman={_have_spearman}, RMST={_have_rmst}, Adjacency={_have_adjacency}")

if not _have_ecm:
    print("WARNING: all_results (ECM-restricted DE) not found. Run ECM signature cell first.")
if not _have_full:
    print("WARNING: all_results_full (full transcriptome DE) not found. Run full transcriptome cell first.")

rows = []

for nh in sorted(nhood_counts.index):
    nh_str = str(nh)
    row = {"neighborhood": nh_str}

    # Spot counts
    n_spots = int(nhood_counts.get(nh, 0))
    row["n_spots"] = n_spots
    row["pct_total"] = round(n_spots / total_spots * 100, 2)

    # Top 10 cell types
    ct_row = None
    if nh in means.index:
        ct_row = means.loc[nh]
    elif nh_str in means.index:
        ct_row = means.loc[nh_str]

    if ct_row is not None:
        top_ct = ct_row.sort_values(ascending=False).head(10)
        row["top10_celltypes"] = "; ".join([f"{ct} ({v:.1%})" for ct, v in top_ct.items()])
    else:
        row["top10_celltypes"] = ""

    # Spearman
    row["spearman_rho"] = ""
    row["spearman_p"] = ""
    row["spearman_fdr"] = ""
    if _have_spearman:
        sp = df_res[df_res["neighborhood"] == nh_str]
        if len(sp):
            row["spearman_rho"] = round(float(sp.iloc[0]["rho"]), 4)
            row["spearman_p"] = round(float(sp.iloc[0]["pval"]), 4)
            if "fdr" in sp.columns:
                row["spearman_fdr"] = round(float(sp.iloc[0]["fdr"]), 4)

    # RMST
    row["rmst_diff_months"] = ""
    row["rmst_p_adj"] = ""
    row["rmst_p_unadj"] = ""
    row["rmst_high_yr"] = ""
    row["rmst_low_yr"] = ""
    if _have_rmst:
        rm = df_rmst[df_rmst["neighborhood"] == nh_str]
        if len(rm):
            row["rmst_diff_months"] = round(float(rm.iloc[0]["diff_months"]), 2)
            row["rmst_p_adj"] = round(float(rm.iloc[0]["p"]), 4)
            row["rmst_p_unadj"] = round(float(rm.iloc[0]["p_unadj"]), 4)
            row["rmst_high_yr"] = round(float(rm.iloc[0]["rmst_high"]), 3)
            row["rmst_low_yr"] = round(float(rm.iloc[0]["rmst_low"]), 3)

    # ECM-restricted signatures (from all_results)
    row["ecm_sig_top5"] = ""
    row["ecm_sig_top10"] = ""
    if _have_ecm:
        for r in all_results:
            if str(r["neighborhood"]) == nh_str:
                row["ecm_sig_top5"] = "; ".join(r["top5"]) if r["top5"] else "---"
                row["ecm_sig_top10"] = "; ".join(r["top10"]) if r["top10"] else "---"
                break

    # Full transcriptome signatures (from all_results_full)
    row["full_sig_top5_up"] = ""
    row["full_sig_top10_up"] = ""
    row["full_sig_top5_down"] = ""
    if _have_full:
        for r in all_results_full:
            if str(r["neighborhood"]) == nh_str:
                row["full_sig_top5_up"] = "; ".join(r["top5"]) if r["top5"] else "---"
                row["full_sig_top10_up"] = "; ".join(r["top10"]) if r["top10"] else "---"
                row["full_sig_top5_down"] = "; ".join(r["top5_down"]) if r["top5_down"] else "---"
                break

    # Spatial adjacency
    row["spatial_co_occurring"] = ""
    row["spatial_avoidant"] = ""
    if _have_adjacency:
        nh_pairs = pairs_df[
            (pairs_df["nhood_A"].astype(str) == nh_str) |
            (pairs_df["nhood_B"].astype(str) == nh_str)
        ].copy()
        nh_pairs["partner"] = nh_pairs.apply(
            lambda x: str(x["nhood_B"]) if str(x["nhood_A"]) == nh_str else str(x["nhood_A"]),
            axis=1
        )

        co_occur = nh_pairs[nh_pairs["log2_enrichment"] > 0].sort_values("log2_enrichment", ascending=False)
        avoidant = nh_pairs[nh_pairs["log2_enrichment"] < 0].sort_values("log2_enrichment", ascending=True)

        co_str = "; ".join([
            f"N{r['partner']} ({r['log2_enrichment']:.2f}, p={r['p_value']:.3g})"
            for _, r in co_occur.head(3).iterrows()
        ])
        av_str = "; ".join([
            f"N{r['partner']} ({r['log2_enrichment']:.2f}, p={r['p_value']:.3g})"
            for _, r in avoidant.head(3).iterrows()
        ])

        row["spatial_co_occurring"] = co_str if co_str else "---"
        row["spatial_avoidant"] = av_str if av_str else "---"

    rows.append(row)

summary = pd.DataFrame(rows)

col_order = [
    "neighborhood", "n_spots", "pct_total",
    "top10_celltypes",
    "spearman_rho", "spearman_p", "spearman_fdr",
    "rmst_diff_months", "rmst_p_adj", "rmst_p_unadj", "rmst_high_yr", "rmst_low_yr",
    "ecm_sig_top5", "ecm_sig_top10",
    "full_sig_top5_up", "full_sig_top10_up", "full_sig_top5_down",
    "spatial_co_occurring", "spatial_avoidant",
]
col_order = [c for c in col_order if c in summary.columns]
summary = summary[col_order]

summary.to_csv("ECM_neighborhood_summary.csv", index=False)

print(f"{'='*120}")
print("ECM NEIGHBORHOOD SUMMARY")
print(f"{'='*120}")
for _, row in summary.iterrows():
    print(f"\n--- Neighborhood {row['neighborhood']} ---")
    print(f"  Spots: {row['n_spots']:,} ({row['pct_total']}%)")
    ct = row.get("top10_celltypes", "")
    if ct:
        print(f"  Top cell types: {ct[:120]}...")
    if row.get("spearman_rho", "") != "":
        print(f"  Spearman vs OS: rho={row['spearman_rho']}, p={row['spearman_p']}, fdr={row.get('spearman_fdr', '')}")
    if row.get("rmst_diff_months", "") != "":
        print(f"  RMST: {row['rmst_diff_months']:+.1f} months (p_adj={row['rmst_p_adj']}, p_unadj={row['rmst_p_unadj']})")
        print(f"    High={row['rmst_high_yr']} yr, Low={row['rmst_low_yr']} yr")
    if row.get("ecm_sig_top5", ""):
        print(f"  ECM sig (5): {row['ecm_sig_top5']}")
        print(f"  ECM sig (10): {row['ecm_sig_top10']}")
    if row.get("full_sig_top5_up", ""):
        print(f"  Full sig UP (5): {row['full_sig_top5_up']}")
        print(f"  Full sig UP (10): {row['full_sig_top10_up']}")
        print(f"  Full sig DOWN (5): {row['full_sig_top5_down']}")
    if row.get("spatial_co_occurring", ""):
        print(f"  Co-occurring: {row['spatial_co_occurring']}")
        print(f"  Avoidant: {row['spatial_avoidant']}")

print(f"\nSaved: ECM_neighborhood_summary.csv")

### Plot examples

In [ ]:
import pandas as pd
import numpy as np

nhood_key = "ECM_neighborhood"
TARGET_NHOOD = 3        # use int, not string
TOP_N = 12
min_spots = 100

sdf = adata.obs[["sample", nhood_key]].dropna().copy()

# Check what the actual values look like
print(f"Dtype: {sdf[nhood_key].dtype}")
print(f"Unique values (first 15): {sorted(sdf[nhood_key].unique())[:15]}")

# Convert consistently to int
sdf[nhood_key] = sdf[nhood_key].astype(float).astype(int)

counts = sdf.groupby(["sample", nhood_key]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1) >= min_spots]
fracs = counts.div(counts.sum(axis=1), axis=0)

print(f"\nColumn dtype: {fracs.columns.dtype}")
print(f"Columns: {list(fracs.columns)}")

if TARGET_NHOOD not in fracs.columns:
    raise ValueError(f"Neighborhood {TARGET_NHOOD} not found. Available: {list(fracs.columns)}")

ranking = fracs[TARGET_NHOOD].sort_values(ascending=False)
top = ranking.head(TOP_N)

print(f"\nTop {TOP_N} samples for Neighborhood {TARGET_NHOOD}:")
print(f"{'Sample':<25} {'Fraction':>10} {'N spots (nhood)':>15} {'N spots (total)':>15}")
print("-" * 70)
for sid, frac_val in top.items():
    n_nh = int(counts.loc[sid, TARGET_NHOOD])
    n_tot = int(counts.loc[sid].sum())
    print(f"{sid:<25} {frac_val:>10.3f} {n_nh:>15,} {n_tot:>15,}")

In [ ]:
import pandas as pd
import numpy as np

# ===== USER CONFIG =====
TARGET_CELLTYPE = "Malignant_MES1"
CELLTYPE_LEVEL = "celltype_level4_curated"   # celltype_level4/3/2_curated
TOP_N = 12
min_spots = 100
# ========================

c2l_prefix = "meanscell_abundance_w_sf_"
celltype_key = adata.uns["celltype_annotation_key"]

if CELLTYPE_LEVEL not in celltype_key.columns:
    raise ValueError(f"{CELLTYPE_LEVEL} not in annotation key. Available: {list(celltype_key.columns)}")

# Get C2L columns and map to curated names at chosen level
c2l_cols = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])

lvl4_in_key = lvl4_names.intersection(celltype_key.index)
curated_map = celltype_key.loc[lvl4_in_key, CELLTYPE_LEVEL].astype(str).fillna("Unknown")

# Sum C2L abundances by curated name
counts_raw = adata.obs[["sample"] + [c2l_prefix + n for n in lvl4_in_key]].copy()
counts_raw.columns = ["sample"] + list(lvl4_in_key)
counts_curated = counts_raw.groupby("sample").sum()
counts_curated = counts_curated.T.groupby(curated_map).sum().T

# Normalize to fractions per sample
row_sums = counts_curated.sum(axis=1)
counts_curated = counts_curated.loc[row_sums >= min_spots]
fracs = counts_curated.div(counts_curated.sum(axis=1), axis=0)

available = sorted(fracs.columns)
print(f"Level: {CELLTYPE_LEVEL}")
print(f"Available cell types ({len(available)}): {available[:20]}{'...' if len(available) > 20 else ''}")

if TARGET_CELLTYPE not in fracs.columns:
    # Fuzzy match suggestion
    matches = [c for c in available if TARGET_CELLTYPE.lower() in c.lower()]
    raise ValueError(
        f"'{TARGET_CELLTYPE}' not found.\n"
        f"Possible matches: {matches if matches else 'None'}\n"
        f"All options: {available}"
    )

ranking = fracs[TARGET_CELLTYPE].sort_values(ascending=False)
top = ranking.head(TOP_N)

print(f"\nTop {TOP_N} samples for {TARGET_CELLTYPE} ({CELLTYPE_LEVEL}):")
print(f"{'Sample':<25} {'Fraction':>10} {'Abundance (sum)':>16} {'Total abundance':>16}")
print("-" * 72)
for sid, frac_val in top.items():
    ct_sum = float(counts_curated.loc[sid, TARGET_CELLTYPE])
    total = float(counts_curated.loc[sid].sum())
    print(f"{sid:<25} {frac_val:>10.3f} {ct_sum:>16.1f} {total:>16.1f}")

In [ ]:
# Cell 2: Plot ECM neighborhoods on H&E

import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import PatchCollection
from matplotlib.colors import to_rgb, to_hex
from matplotlib.lines import Line2D
from PIL import Image
import s3fs
import openslide

# ===== USER CONFIG =====
sample_id = "ER_G_666a_vis"
ecm_neighborhoods = None          # None for all, or [1, 2, 3] for specific
highlight_nhood = None            # int or None; if set, other nhoods are gray

figsize = (14, 12)
spot_alpha = 0.9
he_alpha = 0.5
margin_px = 800
max_view_px = 6000
spot_diam_um = 55.0
fallback_mpp = 0.5
title = None
title_fontsize = 14

show_legend = True                # True/False to toggle legend
show_title = True                 # True/False to toggle title

# Output
save_png = True
save_dpi = 200
jpeg_quality = 92
output_format = "png"             # "png" or "jpg"
# ========================

ECM_ANNOTATIONS_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

S3_ALIAS = "pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias"
VISIUM_PREFIX = {
    "CH": "b5bd8840-cc6e-4de2-a78a-22315e9d1734",
    "ER": "40ce7e61-5ff6-467e-9972-0c85705f086c",
    "CR": "f0415978-2385-4067-b215-13e699a0f7a7",
}
ALIGNED_DIRNAME = "aligned_features_DinoV2ViTBaseDistilled_aquavit"

ECM_COLORS = {
    0: "#377EB8",   1: "#FF7F00",   2: "#4DAF4A",   3: "#E41A1C",   4: "#984EA3",
    5: "#8B4513",   6: "#F198B3",   7: "#808080",   8: "#9ACD32",   9: "#008B8B",
}
GRAY = "#C0C0C0"

# --- Load data ---
s3 = s3fs.S3FileSystem(anon=False)
center = sample_id[:2]
prefix = VISIUM_PREFIX[center]

def s3key(uri):
    return uri.replace("s3://", "")

coords_uri = f"s3://{S3_ALIAS}/{prefix}/v7/image_features/{ALIGNED_DIRNAME}/{sample_id}/filtered_coords.csv"
with s3.open(s3key(coords_uri), "rb") as f:
    coords_df = pd.read_csv(f)
coords_df["barcode"] = coords_df["barcode"].astype(str)

he_uri = f"s3://{S3_ALIAS}/{prefix}/v7/converted_he/{sample_id}.tif"
tmp = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
tmp.close()
s3.get(s3key(he_uri), tmp.name)
slide = openslide.OpenSlide(tmp.name)

ecm_df = pd.read_csv(ECM_ANNOTATIONS_PATH)
ecm_sample = ecm_df[ecm_df["sample"] == sample_id].copy()
ecm_sample["barcode"] = ecm_sample["spot_id"].str.replace(f"{sample_id}_", "", regex=False)
barcode_to_ecm = {}
for _, row in ecm_sample.iterrows():
    try:
        barcode_to_ecm[row["barcode"]] = int(float(row["ECM_neighborhood"]))
    except (ValueError, TypeError):
        continue

# --- Compute view region ---
x_full = coords_df["pxl_col_in_fullres"].values
y_full = coords_df["pxl_row_in_fullres"].values
barcodes = coords_df["barcode"].values

x0 = max(0, int(x_full.min() - margin_px))
y0 = max(0, int(y_full.min() - margin_px))
x1 = int(x_full.max() + margin_px)
y1 = int(y_full.max() + margin_px)
w, h = x1 - x0, y1 - y0

scale = max(w / max_view_px, h / max_view_px, 1.0)
level = slide.get_best_level_for_downsample(scale)
ds = slide.level_downsamples[level]

region = slide.read_region((x0, y0), level, (int(w / ds), int(h / ds))).convert("RGB")
img = np.asarray(region)

mpp = float(slide.properties.get("openslide.mpp-x", fallback_mpp))
r = (spot_diam_um / 2) / mpp / ds

x = (x_full - x0) / ds
y = (y_full - y0) / ds
n_spots = len(x)

# --- Build spot colors ---
if ecm_neighborhoods is None:
    ecm_neighborhoods = list(range(10))

rgba = np.zeros((n_spots, 4))
for i, bc in enumerate(barcodes):
    ecm = barcode_to_ecm.get(bc, -1)
    if ecm < 0:
        continue
    if highlight_nhood is not None:
        if ecm == highlight_nhood:
            rgba[i, :3] = to_rgb(ECM_COLORS[ecm])
            rgba[i, 3] = spot_alpha
        elif ecm in ecm_neighborhoods:
            rgba[i, :3] = to_rgb(GRAY)
            rgba[i, 3] = spot_alpha * 0.4
    else:
        if ecm in ecm_neighborhoods:
            rgba[i, :3] = to_rgb(ECM_COLORS[ecm])
            rgba[i, 3] = spot_alpha

# --- Plot ---
if show_legend:
    fig, (ax, ax_legend) = plt.subplots(1, 2, figsize=(figsize[0] + 3, figsize[1]),
                                         gridspec_kw={"width_ratios": [figsize[0], 2.5],
                                                      "wspace": 0.02})
else:
    fig, ax = plt.subplots(figsize=figsize)

ax.imshow(img, alpha=he_alpha)

patches = [Circle((x[i], y[i]), r) for i in range(n_spots) if rgba[i, 3] > 0]
rgba_visible = rgba[rgba[:, 3] > 0]
if len(patches) > 0:
    pc = PatchCollection(patches, linewidth=0)
    pc.set_facecolors(rgba_visible)
    ax.add_collection(pc)

ax.axis("off")
ax.grid(False)

if show_title:
    if title is None:
        if highlight_nhood is not None:
            auto_title = f"{sample_id} — ECM Neighborhood {highlight_nhood}"
        else:
            auto_title = f"{sample_id} — ECM Neighborhoods"
    else:
        auto_title = title
    ax.set_title(auto_title, fontsize=title_fontsize, fontweight="bold", pad=10)

# Legend in separate panel
if show_legend:
    if highlight_nhood is not None:
        legend_nhoods = [highlight_nhood]
    else:
        legend_nhoods = sorted(ecm_neighborhoods)

    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=ECM_COLORS[n],
                      markersize=12, markeredgewidth=0, label=f"N{n}")
               for n in legend_nhoods]
    ax_legend.legend(handles=handles, loc="center left", frameon=False,
                     fontsize=11, title="ECM\nNeighborhood", title_fontsize=12,
                     handletextpad=0.5, labelspacing=0.6)
    ax_legend.axis("off")
    ax_legend.grid(False)

plt.tight_layout()

# --- Save ---
if save_png:
    out_path = f"{sample_id}_ECM"
    if highlight_nhood is not None:
        out_path += f"_N{highlight_nhood}"

    if output_format == "jpg":
        out_path += ".jpg"
        fig.savefig(out_path, dpi=save_dpi, bbox_inches="tight", pad_inches=0.1,
                    pil_kwargs={"quality": jpeg_quality, "optimize": True})
    else:
        out_path += ".png"
        fig.savefig(out_path, dpi=save_dpi, bbox_inches="tight", pad_inches=0.1)
        img_pil = Image.open(out_path)
        img_pil.save(out_path, optimize=True)

    import os
    fsize_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f"Saved: {out_path} ({fsize_mb:.1f} MB)")

plt.show()
slide.close()

### Compositional ECM category analysis

In [ ]:
# %% [markdown]
# ### Matrisome category analysis per TIL zone (absolute and compositional)
#
# For each ECM category:
#   - absolute expression: sum of category gene expression per spot, averaged per sample × zone
#   - compositional fraction: category sum / total matrisome sum, per spot, averaged per sample × zone
# Per-sample values then tested between TIL-high and TIL-low using Mann-Whitney U.

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.sparse import issparse
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Config
LAYER         = "log_norm_count"
ZONES         = ["TIL+", "BZ1", "BZ2", "TIL-"]
MIN_SPOTS     = 20
FDR_METHOD    = "fdr_bh"

# Category → gene mapping (extracted from the curated brain matrisome panel)
CATEGORY_GENES = {
    "Collagens": [
        "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
        "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
        "COL7A1", "COL8A1", "COL11A1", "COL12A1",
        "COL14A1", "COL15A1", "COL18A1", "COL10A1",
    ],
    "ECM glycoproteins": [
        "FN1", "FBN1", "FBN2",
        "TNC", "TNR", "TNXB",
        "SPARC", "POSTN", "SPP1", "EFEMP1",
        "LAMA1", "LAMA2", "LAMA4", "LAMA5",
        "LAMB1", "LAMB2", "LAMB3",
        "LAMC1", "LAMC3",
        "THBS1", "THBS2", "THBS4",
        "VTN", "NID1", "NID2",
        "FBLN1", "FBLN2", "FBLN5",
        "MMRN2", "VWA1", "TGFBI", "LTBP1",
        "CHI3L1", "LGALS1",
        "CCN1", "CCN2", "EMILIN1",
    ],
    "Proteoglycans": [
        "VCAN", "ACAN", "BCAN", "NCAN",
        "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN",
        "HSPG2", "AGRN",
        "GPC1", "GPC2", "GPC3",
        "SDC1", "SDC4",
        "CSPG4", "CSPG5", "PTPRZ1",
        "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",
    ],
    "Proteases": [
        "MMP2", "MMP7", "MMP9", "MMP11", "MMP12", "MMP13", "MMP14", "MMP16",
        "ADAMTS1", "ADAMTS4", "ADAMTS5",
        "ADAM8", "HPSE", "PLAU", "PLAT", "CTSB",
    ],
    "Protease inhibitors": [
        "TIMP1", "TIMP2", "TIMP3", "SERPINE1",
    ],
    "Crosslinking / collagen maturation": [
        "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4",
        "TGM2",
        "P4HA1", "P4HA2",
        "PLOD1", "PLOD2",
        "BMP1", "PCOLCE",
    ],
    "Hyaluronan metabolism": [
        "HAS2", "HAS3", "HYAL2",
    ],
    "Other secreted ECM factors": [
        "IGFBP5", "SRGN",
    ],
}
CATEGORY_ORDER = list(CATEGORY_GENES.keys())

# Map each gene to its category, restricted to genes present in adata
gene_to_cat = {}
for cat, gs in CATEGORY_GENES.items():
    for g in gs:
        if g in adata.var_names:
            gene_to_cat[g] = cat

all_ecm_genes = list(gene_to_cat.keys())
print(f"ECM genes available: {len(all_ecm_genes)} of {sum(len(v) for v in CATEGORY_GENES.values())} in panel")
for cat in CATEGORY_ORDER:
    n = sum(1 for g in CATEGORY_GENES[cat] if g in gene_to_cat)
    print(f"  {cat}: {n}/{len(CATEGORY_GENES[cat])}")

# Pull expression matrix for the ECM genes
gene_idx = adata.var_names.get_indexer(all_ecm_genes)
X = adata.layers[LAYER][:, gene_idx]
if issparse(X):
    X = X.toarray()
gene_df = pd.DataFrame(X, index=adata.obs_names, columns=all_ecm_genes)

# Per-spot category sums (absolute) and total matrisome sum
cat_sums = pd.DataFrame(index=adata.obs_names, columns=CATEGORY_ORDER, dtype=float)
for cat in CATEGORY_ORDER:
    genes_in_cat = [g for g in CATEGORY_GENES[cat] if g in gene_to_cat]
    if genes_in_cat:
        cat_sums[cat] = gene_df[genes_in_cat].sum(axis=1)
    else:
        cat_sums[cat] = 0.0
total_matrisome = cat_sums.sum(axis=1).replace(0, np.nan)

# Per-spot compositional fractions
cat_fracs = cat_sums.div(total_matrisome, axis=0).fillna(0.0)

# Attach spot-level annotations
spot_df = pd.DataFrame({
    "sample":    obs["sample"].values,
    "TIL_zone":  obs["TIL_zone"].values,
    "TIL_group": obs["TIL_group"].values,
}, index=adata.obs_names)

# Filter spots with valid annotations and nonzero matrisome
valid = (
    spot_df["TIL_zone"].isin(ZONES)
    & spot_df["TIL_group"].isin(["TIL_high", "TIL_low"])
    & total_matrisome.notna()
)
print(f"\nValid spots: {valid.sum():,} of {len(spot_df):,}")

cat_sums_v  = cat_sums[valid]
cat_fracs_v = cat_fracs[valid]
spot_df_v   = spot_df[valid]
total_v     = total_matrisome[valid]

# =====================================================================
# Per-sample summaries (mean across spots within each sample × zone)
# =====================================================================
def per_sample_means(values_df, label):
    """Average values per (sample, zone). Returns long-format DataFrame."""
    work = values_df.copy()
    work["sample"]    = spot_df_v["sample"].values
    work["TIL_zone"]  = spot_df_v["TIL_zone"].values
    work["TIL_group"] = spot_df_v["TIL_group"].values

    # Min spots per (sample, zone) filter
    counts = work.groupby(["sample", "TIL_zone"], observed=True).size().reset_index(name="n")
    keep = counts[counts["n"] >= MIN_SPOTS][["sample", "TIL_zone"]]
    work = work.merge(keep, on=["sample", "TIL_zone"])

    long = work.melt(
        id_vars=["sample", "TIL_zone", "TIL_group"],
        value_vars=CATEGORY_ORDER,
        var_name="category", value_name=label,
    )
    summary = (
        long.groupby(["sample", "TIL_zone", "TIL_group", "category"], observed=True)
            [label].mean().reset_index()
    )
    return summary

abs_per_sample  = per_sample_means(cat_sums_v,  "value")
frac_per_sample = per_sample_means(cat_fracs_v, "value")

# Also total matrisome per sample × zone for the absolute summary plot
total_work = pd.DataFrame({
    "sample":    spot_df_v["sample"].values,
    "TIL_zone":  spot_df_v["TIL_zone"].values,
    "TIL_group": spot_df_v["TIL_group"].values,
    "total":     total_v.values,
})
total_counts = total_work.groupby(["sample", "TIL_zone"], observed=True).size().reset_index(name="n")
keep_total   = total_counts[total_counts["n"] >= MIN_SPOTS][["sample", "TIL_zone"]]
total_work   = total_work.merge(keep_total, on=["sample", "TIL_zone"])
total_per_sample = (
    total_work.groupby(["sample", "TIL_zone", "TIL_group"], observed=True)
              ["total"].mean().reset_index()
)

# =====================================================================
# Statistical tests per (zone × category)
# =====================================================================
def test_per_zone_category(df, value_col):
    """Mann-Whitney U per (zone × category), TIL_high vs TIL_low. BH-FDR across all tests."""
    records = []
    for zone in ZONES:
        for cat in CATEGORY_ORDER:
            h = df[
                (df["TIL_zone"] == zone)
                & (df["category"] == cat)
                & (df["TIL_group"] == "TIL_high")
            ][value_col].dropna().values
            l = df[
                (df["TIL_zone"] == zone)
                & (df["category"] == cat)
                & (df["TIL_group"] == "TIL_low")
            ][value_col].dropna().values
            if len(h) < 3 or len(l) < 3:
                records.append({
                    "zone": zone, "category": cat, "n_high": len(h), "n_low": len(l),
                    "median_high": np.nan, "median_low": np.nan, "p": np.nan,
                })
                continue
            try:
                _, p = mannwhitneyu(h, l, alternative="two-sided")
            except ValueError:
                p = np.nan
            records.append({
                "zone": zone, "category": cat,
                "n_high": len(h), "n_low": len(l),
                "median_high": float(np.median(h)),
                "median_low":  float(np.median(l)),
                "diff_median": float(np.median(h)) - float(np.median(l)),
                "p": p,
            })
    res = pd.DataFrame(records)
    mask = res["p"].notna()
    res["fdr"] = np.nan
    if mask.any():
        _, fdr, _, _ = multipletests(res.loc[mask, "p"], method=FDR_METHOD)
        res.loc[mask, "fdr"] = fdr
    return res

abs_stats  = test_per_zone_category(abs_per_sample,  "value")
frac_stats = test_per_zone_category(frac_per_sample, "value")

# Print summaries
def print_results(stats, label):
    print(f"\n{'='*100}")
    print(f"{label}")
    print(f"{'='*100}")
    for zone in ZONES:
        z = stats[stats["zone"] == zone].copy()
        print(f"\n{zone}:")
        fmt = z[["category", "n_high", "n_low",
                 "median_high", "median_low", "diff_median", "p", "fdr"]].copy()
        for c in ["median_high", "median_low", "diff_median"]:
            fmt[c] = fmt[c].round(4)
        for c in ["p", "fdr"]:
            fmt[c] = fmt[c].apply(lambda v: f"{v:.2e}" if pd.notna(v) else "—")
        print(fmt.to_string(index=False))

print_results(abs_stats, "ABSOLUTE matrisome expression (TIL-high vs TIL-low, per zone × category)")
print_results(frac_stats, "COMPOSITIONAL matrisome fraction (TIL-high vs TIL-low, per zone × category)")

# Save tables
abs_stats.to_csv("matrisome_absolute_per_zone_category.csv", index=False)
frac_stats.to_csv("matrisome_compositional_per_zone_category.csv", index=False)
abs_per_sample.to_csv("matrisome_absolute_per_sample.csv", index=False)
frac_per_sample.to_csv("matrisome_compositional_per_sample.csv", index=False)
total_per_sample.to_csv("matrisome_total_per_sample.csv", index=False)
print("\nSaved tables: matrisome_absolute_per_zone_category.csv, "
      "matrisome_compositional_per_zone_category.csv, and per-sample tables.")

# =====================================================================
# Plot 1: Total matrisome expression per zone (absolute), TIL-high vs TIL-low
# =====================================================================
COL_HIGH_FACE = "#f4c8c1"
COL_HIGH_EDGE = "#b2182b"
COL_LOW_FACE  = "#c5d9ec"
COL_LOW_EDGE  = "#2166ac"

fig, ax = plt.subplots(figsize=(7, 5))
group_w = 0.40
for zi, zone in enumerate(ZONES):
    for grp, off, face, edge in [
        ("TIL_high", -group_w/2, COL_HIGH_FACE, COL_HIGH_EDGE),
        ("TIL_low",   group_w/2, COL_LOW_FACE,  COL_LOW_EDGE),
    ]:
        sub = total_per_sample[
            (total_per_sample["TIL_zone"] == zone)
            & (total_per_sample["TIL_group"] == grp)
        ]
        vals = sub["total"].dropna().values
        if len(vals) < 3:
            continue
        bp = ax.boxplot(vals, positions=[zi + off], widths=group_w * 0.85,
                        patch_artist=True, showfliers=False, zorder=2)
        for box in bp["boxes"]: box.set(facecolor=face, edgecolor=edge, linewidth=0.9)
        for w in bp["whiskers"]: w.set(color="#444444", linewidth=0.6)
        for c in bp["caps"]:     c.set(color="#444444", linewidth=0.6)
        for m in bp["medians"]:  m.set(color="black", linewidth=1.1)
        jitter = (np.random.RandomState(42).rand(len(vals)) - 0.5) * (group_w * 0.4)
        for v, j in zip(vals, jitter):
            ax.scatter(zi + off + j, v, s=4, color=edge, alpha=0.45,
                       zorder=3, edgecolors="none")

ax.set_xticks(range(len(ZONES)))
ax.set_xticklabels(ZONES, fontsize=11)
ax.set_ylabel("Per-patient mean total matrisome\nexpression (log-normalized sum)", fontsize=11)
ax.grid(False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.legend(handles=[
    Patch(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
    Patch(facecolor=COL_LOW_FACE,  edgecolor=COL_LOW_EDGE,  label="TIL-low"),
], loc="upper right", frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig("matrisome_total_per_zone.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: matrisome_total_per_zone.png")

# =====================================================================
# Plot 2: Compositional fraction heatmap — diff_median (TIL_high - TIL_low) per zone × category
# Significance stars from FDR-adjusted Mann-Whitney p-values
# =====================================================================
heatmap = frac_stats.pivot_table(
    index="category", columns="zone", values="diff_median",
).reindex(index=CATEGORY_ORDER, columns=ZONES)
fdr_mat = frac_stats.pivot_table(
    index="category", columns="zone", values="fdr",
).reindex(index=CATEGORY_ORDER, columns=ZONES)

vmax = np.nanmax(np.abs(heatmap.values))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heatmap.values, aspect="auto", cmap="RdBu_r",
               vmin=-vmax, vmax=vmax, interpolation="none")

ax.set_xticks(range(len(ZONES)))
ax.set_xticklabels(ZONES, fontsize=11)
ax.set_yticks(range(len(CATEGORY_ORDER)))
ax.set_yticklabels(CATEGORY_ORDER, fontsize=11)
ax.set_xlabel("TIL zone", fontsize=11, labelpad=8)
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

# Annotate with diff and significance stars
def stars(q):
    if pd.isna(q):  return ""
    if q < 0.001:   return "***"
    if q < 0.01:    return "**"
    if q < 0.05:    return "*"
    return ""
for i, cat in enumerate(CATEGORY_ORDER):
    for j, zone in enumerate(ZONES):
        v = heatmap.loc[cat, zone]
        q = fdr_mat.loc[cat, zone]
        if pd.notna(v):
            star = stars(q)
            label = f"{v:+.3f}\n{star}" if star else f"{v:+.3f}"
            ax.text(j, i, label, ha="center", va="center",
                    fontsize=9,
                    color="white" if abs(v) > vmax * 0.55 else "#222222")

cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, shrink=0.8)
cbar.set_label("Compositional fraction difference\n(TIL-high − TIL-low)", fontsize=10)
cbar.ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig("matrisome_compositional_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: matrisome_compositional_heatmap.png")

# =====================================================================
# Plot 3: Per-category boxplots showing compositional fraction across zones
# One panel per ECM category
# =====================================================================
n_cats = len(CATEGORY_ORDER)
n_cols = 4
n_rows = int(np.ceil(n_cats / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for ai, cat in enumerate(CATEGORY_ORDER):
    ax = axes[ai]
    for zi, zone in enumerate(ZONES):
        for grp, off, face, edge in [
            ("TIL_high", -group_w/2, COL_HIGH_FACE, COL_HIGH_EDGE),
            ("TIL_low",   group_w/2, COL_LOW_FACE,  COL_LOW_EDGE),
        ]:
            sub = frac_per_sample[
                (frac_per_sample["TIL_zone"] == zone)
                & (frac_per_sample["TIL_group"] == grp)
                & (frac_per_sample["category"] == cat)
            ]
            vals = sub["value"].dropna().values
            if len(vals) < 3:
                continue
            bp = ax.boxplot(vals, positions=[zi + off], widths=group_w * 0.85,
                            patch_artist=True, showfliers=False, zorder=2)
            for box in bp["boxes"]: box.set(facecolor=face, edgecolor=edge, linewidth=0.8)
            for w in bp["whiskers"]: w.set(color="#444444", linewidth=0.5)
            for c in bp["caps"]:     c.set(color="#444444", linewidth=0.5)
            for m in bp["medians"]:  m.set(color="black", linewidth=1.0)
            jitter = (np.random.RandomState(42).rand(len(vals)) - 0.5) * (group_w * 0.4)
            for v, j in zip(vals, jitter):
                ax.scatter(zi + off + j, v, s=3, color=edge, alpha=0.4,
                           zorder=3, edgecolors="none")
    ax.set_xticks(range(len(ZONES)))
    ax.set_xticklabels(ZONES, fontsize=9)
    ax.set_ylabel("Fraction", fontsize=9)
    ax.set_title(cat, fontsize=10, loc="left")
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(labelsize=8)

# Hide unused axes
for ai in range(n_cats, len(axes)):
    axes[ai].set_visible(False)

axes[0].legend(handles=[
    Patch(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
    Patch(facecolor=COL_LOW_FACE,  edgecolor=COL_LOW_EDGE,  label="TIL-low"),
], loc="upper right", frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig("matrisome_compositional_per_category.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: matrisome_compositional_per_category.png")